# Gasoil–Brent Feedback Loop: Quantitative Evidence from Futures Markets

**Research Type:** Quantitative Discovery & Hypothesis Testing  
**Primary Data:** ICE Brent Crude (LCO) and ICE Gasoil (LGO) Futures Settlement/Close Prices  
**Date Range:** 2016–2026 (Brent Settle), 2021–2026 (Gasoil Intraday → Daily)  
**Hypothesis Under Investigation:** Does a GO–Brent widening event trigger a systematic feedback loop in crude oil markets?

---

## Research Motivation

A proposed market feedback mechanism suggests:

1. GO–Brent spread widens → refinery margins increase
2. Refineries respond by purchasing more crude → Brent prices rise
3. Rising Brent narrows the spread → demand weakens
4. Cycle repeats

**This notebook does NOT assume this loop exists.** Instead, we let the data reveal what 
relationships, lead-lag effects, feedback mechanisms, and regime dependencies actually exist 
in the historical record.

---

## Setup & Configuration

In [1]:
# ─── Standard Library ───
import warnings
warnings.filterwarnings('ignore')

import os
import re
import json
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict

# ─── Numerical & Data ───
import numpy as np
import pandas as pd
from scipy import stats, signal
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

# ─── Statistical Modeling ───
from statsmodels.tsa.stattools import grangercausalitytests, adfuller, coint
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.api import VAR as VARapi
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm

# ─── Machine Learning ───
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans

# ─── Visualization ───
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import seaborn as sns

# ─── Plot Configuration ───
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
})

# Color palette for consistent styling
COLORS = {
    'brent': '#1f77b4',
    'gasoil': '#ff7f0e',
    'spread': '#2ca02c',
    'event': '#d62728',
    'neutral': '#7f7f7f',
    'highlight': '#9467bd',
    'backwardation': '#00c48f',
    'contango': '#ff4757',
}

DATA_DIR = Path(r"f:\dashboard 2 (2)\dashboard 2\dashboard 2\data")
print(f"Data directory: {DATA_DIR}")
print(f"Files available: {[f.name for f in DATA_DIR.glob('*.csv')]}")

Data directory: f:\dashboard 2 (2)\dashboard 2\dashboard 2\data
Files available: ['CL_data.csv', 'CL_outrights_1min_t.csv', 'HO_data.csv', 'LCOSettle 2.csv', 'LCO_3_year_test.csv', 'LCO_data.csv', 'LGO_data.csv', 'wtcl_lco_outrights_1min.csv']


---
# ANALYSIS 1: Data Inventory & Quality Assessment

Before any analysis, we systematically catalogue every available dataset, identifying 
contracts, maturities, date ranges, missing values, and potential issues.

In [2]:
# ─── Load Brent Settlement Data (Daily, LCOSettle 2.csv) ───
# This file has a paired-column format: (Timestamp, SETTLE) repeated for each contract

def load_brent_settle(filepath):
    """Parse the ICE Brent settlement file with paired Timestamp/Settle columns."""
    with open(filepath, 'r') as f:
        contract_row = f.readline().strip().split(',')
        field_row = f.readline().strip().split(',')

    df_raw = pd.read_csv(filepath, skiprows=2, header=None)

    settle_data = {}
    date_col = None

    for i in range(0, len(contract_row), 2):
        contract_name = contract_row[i].strip()
        match = re.search(r'c(\d+)', contract_name)
        if match:
            c_num = int(match.group(1))
            if date_col is None:
                date_col = df_raw.iloc[:, i]
            settle_data[f'LCO_c{c_num}'] = pd.to_numeric(df_raw.iloc[:, i + 1], errors='coerce')

    result = pd.DataFrame(settle_data)
    dates = pd.to_datetime(date_col, format='%d-%m-%y', errors='coerce')
    result.index = dates
    result.index.name = 'date'
    result = result.sort_index()
    result = result.dropna(how='all')

    return result

brent_settle = load_brent_settle(DATA_DIR / 'LCOSettle 2.csv')
print(f"Brent Settlement Data loaded: {brent_settle.shape}")
print(f"Date range: {brent_settle.index.min().date()} to {brent_settle.index.max().date()}")
print(f"Contracts: {list(brent_settle.columns)}")
brent_settle.head()

Brent Settlement Data loaded: (2712, 31)
Date range: 2016-01-04 to 2026-05-26
Contracts: ['LCO_c1', 'LCO_c2', 'LCO_c3', 'LCO_c4', 'LCO_c5', 'LCO_c6', 'LCO_c7', 'LCO_c8', 'LCO_c9', 'LCO_c10', 'LCO_c11', 'LCO_c12', 'LCO_c13', 'LCO_c14', 'LCO_c15', 'LCO_c16', 'LCO_c17', 'LCO_c18', 'LCO_c19', 'LCO_c20', 'LCO_c21', 'LCO_c22', 'LCO_c23', 'LCO_c24', 'LCO_c25', 'LCO_c26', 'LCO_c27', 'LCO_c28', 'LCO_c29', 'LCO_c30', 'LCO_c31']


,LCO_c1,LCO_c2,LCO_c3,LCO_c4,LCO_c5,LCO_c6,LCO_c7,LCO_c8,LCO_c9,LCO_c10,...,LCO_c22,LCO_c23,LCO_c24,LCO_c25,LCO_c26,LCO_c27,LCO_c28,LCO_c29,LCO_c30,LCO_c31
date,,,,,,,,,,,,,,,,,,,,,
2016-01-04,37.22,37.60,38.47,39.33,40.18,40.98,41.71,42.40,43.03,43.65,...,49.15,49.45,49.81,50.17,50.53,50.89,51.26,51.63,51.90,52.14
2016-01-05,36.42,36.89,37.77,38.65,39.50,40.29,41.02,41.71,42.35,42.98,...,48.45,48.75,49.10,49.45,49.80,50.16,50.52,50.88,51.14,51.38
2016-01-06,34.23,34.63,35.56,36.51,37.43,38.27,39.05,39.79,40.46,41.11,...,46.81,47.13,47.49,47.85,48.21,48.59,48.96,49.33,49.60,49.85
2016-01-07,33.75,34.09,35.01,35.95,36.86,37.68,38.44,39.16,39.83,40.48,...,46.02,46.33,46.69,47.05,47.40,47.78,48.14,48.51,48.78,49.03
2016-01-08,33.55,33.93,34.87,35.84,36.76,37.60,38.36,39.09,39.77,40.43,...,45.90,46.20,46.55,46.90,47.24,47.61,47.97,48.33,48.60,48.84


In [3]:
# ─── Load Gasoil Intraday Data and Resample to Daily ───
# LGO_data.csv is 1-min data; we extract the last price per day per contract

def load_intraday_to_daily(filepath, prefix='LGO'):
    """
    Load 1-minute outright data and resample to daily close.
    Memory-efficient: reads only price columns.
    """
    print(f"Loading {filepath.name} ({filepath.stat().st_size / 1e9:.2f} GB)...")

    # Read header to find weighted_mid columns
    with open(filepath, 'r') as f:
        meta_line = f.readline()
        header_line = f.readline().strip()

    headers = [h.strip() for h in header_line.split(',')]

    # Identify columns
    usecols = [0]  # timestamp
    col_map = {0: 'timestamp'}

    for i, h in enumerate(headers):
        if 'weighted_mid' in h:
            usecols.append(i)
            contract_num = h.split('||')[0].strip()
            col_map[i] = f'{prefix}_{contract_num}'

    # Read CSV in chunks for memory efficiency
    chunks = []
    for chunk in pd.read_csv(filepath, skiprows=1, header=0, chunksize=500_000,
                              low_memory=False):
        # Get timestamp and weighted_mid columns
        ts_col = chunk.columns[0]
        mid_cols = [col for col in chunk.columns if 'weighted_mid' in col]

        sub = chunk[[ts_col] + mid_cols].copy()
        sub[ts_col] = pd.to_datetime(sub[ts_col], utc=True, errors='coerce')
        sub = sub.set_index(ts_col)
        sub.index.name = 'timestamp'

        # Rename columns
        rename_map = {}
        for col in mid_cols:
            contract_num = col.split('||')[0].strip()
            rename_map[col] = f'{prefix}_{contract_num}'
        sub = sub.rename(columns=rename_map)

        # Convert to numeric
        for col in sub.columns:
            sub[col] = pd.to_numeric(sub[col], errors='coerce')

        # Resample to daily (last observation)
        daily_chunk = sub.groupby(sub.index.date).last()
        chunks.append(daily_chunk)

    daily = pd.concat(chunks)
    # If multiple chunks cover the same day, take the last value
    daily = daily.groupby(daily.index).last()
    daily.index = pd.to_datetime(daily.index)
    daily.index.name = 'date'
    daily = daily.sort_index()
    daily = daily.dropna(how='all')

    print(f"  → Resampled to {len(daily)} trading days")
    print(f"  → Date range: {daily.index.min().date()} to {daily.index.max().date()}")
    return daily

gasoil_daily = load_intraday_to_daily(DATA_DIR / 'LGO_data.csv', prefix='LGO')
print(f"\nGasoil Daily Data: {gasoil_daily.shape}")
gasoil_daily.head()

Loading LGO_data.csv (0.51 GB)...


  → Resampled to 1570 trading days
  → Date range: 2021-01-04 to 2026-05-20

Gasoil Daily Data: (1570, 14)


,LGO_c1,LGO_c2,LGO_c3,LGO_c4,LGO_c5,LGO_c6,LGO_c7,LGO_c8,LGO_c9,LGO_c10,LGO_c11,LGO_c12,LGO_c13,LGO_c14
date,,,,,,,,,,,,,,
2021-01-04,414.607143,417.392857,419.750000,421.250000,422.583333,424.000000,425.650000,427.500000,429.000000,430.666667,430.500000,430.285714,NaN,450.142857
2021-01-05,433.200000,436.083333,438.294118,439.571429,440.375000,441.125000,442.500000,443.875000,445.000000,446.500000,446.375000,445.200000,NaN,448.500000
2021-01-06,432.894737,436.144737,438.250000,439.600000,440.428571,441.428571,442.821429,444.321429,445.464286,446.821429,446.571429,445.428571,NaN,447.958333
2021-01-07,435.000000,438.416667,441.375000,442.875000,443.625000,444.666667,445.875000,447.125000,448.125000,449.250000,449.173077,447.923077,NaN,447.958333
2021-01-08,448.666667,452.476190,454.750000,456.227778,456.833333,457.350000,458.333333,459.593750,460.050000,460.833333,460.083333,458.666667,453.666667,462.666667


In [4]:
# ─── Also load Brent intraday → daily for the same period (2021+) ───
# This gives us a consistent Brent daily close aligned with Gasoil daily close

brent_intraday_daily = load_intraday_to_daily(DATA_DIR / 'LCO_data.csv', prefix='LCO_intra')
print(f"\nBrent Intraday→Daily Data: {brent_intraday_daily.shape}")
brent_intraday_daily.head()

Loading LCO_data.csv (0.62 GB)...


  → Resampled to 1572 trading days
  → Date range: 2021-01-04 to 2026-05-22

Brent Intraday→Daily Data: (1572, 17)


,LCO_intra_c1,LCO_intra_c2,LCO_intra_c3,LCO_intra_c4,LCO_intra_c5,LCO_intra_c6,LCO_intra_c7,LCO_intra_c8,LCO_intra_c9,LCO_intra_c10,LCO_intra_c11,LCO_intra_c12,LCO_intra_c13,LCO_intra_c14,LCO_intra_c15,LCO_intra_c16,LCO_intra_c17
date,,,,,,,,,,,,,,,,,
2021-01-04,50.664286,50.744286,50.772857,50.711429,50.580000,50.458571,50.337143,50.195714,50.073333,49.955000,49.830000,49.750000,52.791980,49.620,49.550000,49.483333,49.480
2021-01-05,53.583000,53.496667,53.376667,53.176667,52.925000,52.693333,52.443333,52.220000,52.000000,51.775000,51.515000,51.549231,52.791980,51.050,51.106129,50.877692,50.725
2021-01-06,54.205000,54.062500,53.884000,53.630000,53.335000,53.045000,52.741429,52.451429,52.183333,51.917500,51.605000,51.380000,52.900891,51.030,50.729677,50.802500,50.625
2021-01-07,54.533636,54.449091,54.253333,54.023333,53.705000,53.415000,53.125000,52.816667,52.535000,52.296667,51.975000,51.770000,52.940495,51.430,51.275000,51.206667,51.050
2021-01-08,56.272000,56.116000,55.866000,55.575714,55.221429,54.887143,54.548889,54.240000,53.920000,53.633333,52.076522,50.187895,52.999901,52.512,52.480000,52.375000,52.195


In [5]:
# ─── Comprehensive Data Inventory ───

print("=" * 80)
print("DATA INVENTORY")
print("=" * 80)

datasets = {
    'Brent Settlement (Daily)': brent_settle,
    'Gasoil Daily (from 1-min)': gasoil_daily,
    'Brent Daily (from 1-min)': brent_intraday_daily,
}

inventory = []
for name, df in datasets.items():
    n_contracts = len(df.columns)
    date_min = df.index.min().date()
    date_max = df.index.max().date()
    n_days = len(df)
    pct_missing = (df.isna().sum().sum() / (df.shape[0] * df.shape[1])) * 100

    inventory.append({
        'Dataset': name,
        'Contracts': n_contracts,
        'Start': date_min,
        'End': date_max,
        'Trading Days': n_days,
        'Missing %': f'{pct_missing:.2f}%',
    })

inventory_df = pd.DataFrame(inventory)
print(inventory_df.to_string(index=False))

print("\n" + "=" * 80)
print("CONTRACT DETAILS")
print("=" * 80)

for name, df in datasets.items():
    print(f"\n{name}:")
    print(f"  Columns: {list(df.columns)}")
    print(f"  c1 range: {df.iloc[:, 0].min():.2f} – {df.iloc[:, 0].max():.2f}")

    # Missing values per contract
    missing = df.isna().sum()
    if missing.sum() > 0:
        print(f"  Missing values: {dict(missing[missing > 0])}")
    else:
        print(f"  Missing values: None")

DATA INVENTORY
                  Dataset  Contracts      Start        End  Trading Days Missing %
 Brent Settlement (Daily)         31 2016-01-04 2026-05-26          2712     0.00%
Gasoil Daily (from 1-min)         14 2021-01-04 2026-05-20          1570     0.02%
 Brent Daily (from 1-min)         17 2021-01-04 2026-05-22          1572     0.00%

CONTRACT DETAILS

Brent Settlement (Daily):
  Columns: ['LCO_c1', 'LCO_c2', 'LCO_c3', 'LCO_c4', 'LCO_c5', 'LCO_c6', 'LCO_c7', 'LCO_c8', 'LCO_c9', 'LCO_c10', 'LCO_c11', 'LCO_c12', 'LCO_c13', 'LCO_c14', 'LCO_c15', 'LCO_c16', 'LCO_c17', 'LCO_c18', 'LCO_c19', 'LCO_c20', 'LCO_c21', 'LCO_c22', 'LCO_c23', 'LCO_c24', 'LCO_c25', 'LCO_c26', 'LCO_c27', 'LCO_c28', 'LCO_c29', 'LCO_c30', 'LCO_c31']
  c1 range: 19.33 – 127.98
  Missing values: None

Gasoil Daily (from 1-min):
  Columns: ['LGO_c1', 'LGO_c2', 'LGO_c3', 'LGO_c4', 'LGO_c5', 'LGO_c6', 'LGO_c7', 'LGO_c8', 'LGO_c9', 'LGO_c10', 'LGO_c11', 'LGO_c12', 'LGO_c13', 'LGO_c14']
  c1 range: 414.61 – 1568.50


In [6]:
# ─── Build Unified Daily Dataset ───
# Align Brent and Gasoil on common dates for the overlap period

# For the 2021+ period: use intraday-derived daily close for both
# For 2016-2020: only Brent settle data available (no Gasoil)

# Primary working dataset: overlap period where both Brent & Gasoil available
common_dates = brent_intraday_daily.index.intersection(gasoil_daily.index)

brent_aligned = brent_intraday_daily.loc[common_dates].copy()
gasoil_aligned = gasoil_daily.loc[common_dates].copy()

# Rename for clarity
brent_cols = {col: col.replace('LCO_intra_', 'brent_') for col in brent_aligned.columns}
gasoil_cols = {col: col.replace('LGO_', 'gasoil_') for col in gasoil_aligned.columns}

brent_aligned = brent_aligned.rename(columns=brent_cols)
gasoil_aligned = gasoil_aligned.rename(columns=gasoil_cols)

# Merge
unified = pd.concat([brent_aligned, gasoil_aligned], axis=1)
unified = unified.dropna(subset=['brent_c1', 'gasoil_c1'])

print(f"Unified dataset: {unified.shape[0]} trading days, {unified.shape[1]} columns")
print(f"Date range: {unified.index.min().date()} to {unified.index.max().date()}")
print(f"\nBrent columns: {[c for c in unified.columns if 'brent' in c]}")
print(f"Gasoil columns: {[c for c in unified.columns if 'gasoil' in c]}")

# Also keep the longer Brent settle series for extended analyses
brent_long = brent_settle.copy()
brent_long.columns = [c.replace('LCO_', 'brent_') for c in brent_long.columns]
print(f"\nExtended Brent series: {brent_long.shape[0]} trading days (2016-2026)")

Unified dataset: 1570 trading days, 31 columns
Date range: 2021-01-04 to 2026-05-20

Brent columns: ['brent_c1', 'brent_c2', 'brent_c3', 'brent_c4', 'brent_c5', 'brent_c6', 'brent_c7', 'brent_c8', 'brent_c9', 'brent_c10', 'brent_c11', 'brent_c12', 'brent_c13', 'brent_c14', 'brent_c15', 'brent_c16', 'brent_c17']
Gasoil columns: ['gasoil_c1', 'gasoil_c2', 'gasoil_c3', 'gasoil_c4', 'gasoil_c5', 'gasoil_c6', 'gasoil_c7', 'gasoil_c8', 'gasoil_c9', 'gasoil_c10', 'gasoil_c11', 'gasoil_c12', 'gasoil_c13', 'gasoil_c14']

Extended Brent series: 2712 trading days (2016-2026)


### Interpretation — Data Inventory

**Trader's Perspective:**
- We have ~10 years of Brent settlement data (2016–2026) across 31 maturities — excellent for curve structure analysis
- Gasoil data is available from 2021 onwards (14 contracts) — this limits our GO–Brent analysis to ~5 years
- The 2021+ period includes COVID recovery, Russia-Ukraine war, and subsequent normalization — rich for regime analysis
- The Brent settle data covers additional regime periods (2016 oil crash, 2018-2019 cycle)

**Limitation:** The overlapping period (2021–2026) is relatively short for discovering multi-year cycles. Statistical significance thresholds should be interpreted accordingly.

---
# ANALYSIS 2: Derived Series Construction

We generate a comprehensive set of derived market variables: spreads, returns, volatility 
measures, z-scores, and momentum indicators.

In [7]:
# ─── GO-Brent Spread ───
# Gasoil is in $/mt, Brent in $/bbl. Standard crack spread comparison uses:
# GO-Brent spread = Gasoil($/mt) / 7.45 - Brent($/bbl)  [convert $/mt to $/bbl approx]
# Or simply the raw difference in native units for spread dynamics
# We'll compute both the raw $/mt vs $/bbl spread and the converted spread

GASOIL_BBL_CONVERSION = 7.45  # Approximate barrels per metric ton for gasoil

# Converted GO-Brent spread (both in $/bbl)
unified['go_brent_spread'] = (unified['gasoil_c1'] / GASOIL_BBL_CONVERSION) - unified['brent_c1']

# Also compute for c2, c3, c4 for robustness
for i in [2, 3, 4]:
    b_col = f'brent_c{i}'
    g_col = f'gasoil_c{i}'
    if b_col in unified.columns and g_col in unified.columns:
        unified[f'go_brent_spread_c{i}'] = (unified[g_col] / GASOIL_BBL_CONVERSION) - unified[b_col]

# Spread ratio (dimensionless)
unified['go_brent_ratio'] = (unified['gasoil_c1'] / GASOIL_BBL_CONVERSION) / unified['brent_c1']

print("GO-Brent Spread Statistics ($/bbl equivalent):")
print(unified['go_brent_spread'].describe().round(2))

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

axes[0].plot(unified.index, unified['go_brent_spread'], color=COLORS['spread'], linewidth=0.8)
axes[0].axhline(y=unified['go_brent_spread'].mean(), color='red', linestyle='--', alpha=0.5,
                 label=f"Mean: ${unified['go_brent_spread'].mean():.2f}")
axes[0].fill_between(unified.index, 
                      unified['go_brent_spread'].mean() - 2*unified['go_brent_spread'].std(),
                      unified['go_brent_spread'].mean() + 2*unified['go_brent_spread'].std(),
                      alpha=0.1, color='red', label='±2σ band')
axes[0].set_ylabel('$/bbl')
axes[0].set_title('GO–Brent Spread (Front Month, $/bbl equivalent)')
axes[0].legend()

axes[1].plot(unified.index, unified['go_brent_ratio'], color=COLORS['highlight'], linewidth=0.8)
axes[1].axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)
axes[1].set_ylabel('Ratio')
axes[1].set_title('Gasoil/Brent Price Ratio')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.savefig('go_brent_spread_overview.png', dpi=150, bbox_inches='tight')
plt.show()

GO-Brent Spread Statistics ($/bbl equivalent):
count    1570.00
mean       25.56
std        15.35
min         2.90
25%        16.07
50%        21.94
75%        34.04
max        98.89
Name: go_brent_spread, dtype: float64


In [8]:
# ─── Brent Calendar Spreads ───
# Using settlement data for the full 2016-2026 range

brent_long['M1_M2'] = brent_long['brent_c1'] - brent_long['brent_c2']
brent_long['M2_M3'] = brent_long['brent_c2'] - brent_long['brent_c3']
brent_long['M2_M4'] = brent_long['brent_c2'] - brent_long['brent_c4']
brent_long['M1_M6'] = brent_long['brent_c1'] - brent_long['brent_c6']
brent_long['M1_M12'] = brent_long['brent_c1'] - brent_long['brent_c12']
brent_long['M1_M3'] = brent_long['brent_c1'] - brent_long['brent_c3']
brent_long['M6_M12'] = brent_long['brent_c6'] - brent_long['brent_c12']

# Same for the unified (2021+) dataset
unified['brent_M1_M2'] = unified['brent_c1'] - unified['brent_c2']
unified['brent_M2_M3'] = unified['brent_c2'] - unified['brent_c3']
unified['brent_M2_M4'] = unified['brent_c2'] - unified['brent_c4']
unified['brent_M1_M6'] = unified['brent_c1'] - unified['brent_c6']
unified['brent_M1_M3'] = unified['brent_c1'] - unified['brent_c3']

if 'brent_c12' in unified.columns:
    unified['brent_M1_M12'] = unified['brent_c1'] - unified['brent_c12']

# Gasoil calendar spreads
unified['gasoil_M1_M2'] = unified['gasoil_c1'] - unified['gasoil_c2']
unified['gasoil_M2_M3'] = unified['gasoil_c2'] - unified['gasoil_c3']
if 'gasoil_c6' in unified.columns:
    unified['gasoil_M1_M6'] = unified['gasoil_c1'] - unified['gasoil_c6']

print("Brent Calendar Spread Statistics (2016-2026):")
spread_cols = ['M1_M2', 'M2_M3', 'M2_M4', 'M1_M6', 'M1_M12']
print(brent_long[spread_cols].describe().round(2))

Brent Calendar Spread Statistics (2016-2026):
         M1_M2    M2_M3    M2_M4    M1_M6   M1_M12
count  2712.00  2712.00  2712.00  2712.00  2712.00
mean      0.47     0.42     0.77     1.82     3.17
std       1.19     0.98     1.78     4.09     6.13
min      -3.97    -3.36    -5.99   -13.01   -16.46
25%      -0.15    -0.12    -0.21    -0.40    -0.22
50%       0.37     0.35     0.66     1.56     3.02
75%       0.73     0.63     1.18     2.82     5.03
max      14.38     8.01    13.80    35.40    42.29


In [9]:
# ─── Returns ───

# Daily log returns
unified['brent_ret'] = np.log(unified['brent_c1'] / unified['brent_c1'].shift(1))
unified['gasoil_ret'] = np.log(unified['gasoil_c1'] / unified['gasoil_c1'].shift(1))
unified['spread_chg'] = unified['go_brent_spread'] - unified['go_brent_spread'].shift(1)
unified['spread_ret_pct'] = unified['spread_chg'] / unified['go_brent_spread'].shift(1).abs()

# Brent long series returns
brent_long['brent_ret'] = np.log(brent_long['brent_c1'] / brent_long['brent_c1'].shift(1))
brent_long['M1_M2_chg'] = brent_long['M1_M2'] - brent_long['M1_M2'].shift(1)

# Rolling returns (5d, 10d, 20d)
for window in [5, 10, 20]:
    unified[f'brent_ret_{window}d'] = unified['brent_c1'].pct_change(window)
    unified[f'gasoil_ret_{window}d'] = unified['gasoil_c1'].pct_change(window)
    unified[f'spread_chg_{window}d'] = unified['go_brent_spread'] - unified['go_brent_spread'].shift(window)

print("Daily Return Statistics:")
print(unified[['brent_ret', 'gasoil_ret', 'spread_chg']].describe().round(4))

Daily Return Statistics:
       brent_ret  gasoil_ret  spread_chg
count  1569.0000   1569.0000   1569.0000
mean      0.0005      0.0007      0.0313
std       0.0228      0.0273      2.6928
min      -0.1832     -0.3186    -40.2278
25%      -0.0097     -0.0108     -0.5779
50%       0.0008      0.0007      0.0084
75%       0.0122      0.0130      0.6519
max       0.1423      0.1800     29.5226


In [10]:
# ─── Volatility ───

# Realized volatility (annualized)
for window in [10, 20, 60]:
    unified[f'brent_rvol_{window}d'] = unified['brent_ret'].rolling(window).std() * np.sqrt(252)
    unified[f'gasoil_rvol_{window}d'] = unified['gasoil_ret'].rolling(window).std() * np.sqrt(252)
    unified[f'spread_vol_{window}d'] = unified['spread_chg'].rolling(window).std()

# Brent long volatility
for window in [20, 60]:
    brent_long[f'brent_rvol_{window}d'] = brent_long['brent_ret'].rolling(window).std() * np.sqrt(252)

print("Current Realized Volatility (20d annualized):")
print(f"  Brent: {unified['brent_rvol_20d'].iloc[-1]:.1%}")
print(f"  Gasoil: {unified['gasoil_rvol_20d'].iloc[-1]:.1%}")

Current Realized Volatility (20d annualized):
  Brent: 65.5%
  Gasoil: 51.8%


In [11]:
# ─── Rolling Z-Scores & Percentiles ───

for window in [60, 120, 252]:
    # GO-Brent spread z-score
    roll_mean = unified['go_brent_spread'].rolling(window).mean()
    roll_std = unified['go_brent_spread'].rolling(window).std()
    unified[f'spread_zscore_{window}d'] = (unified['go_brent_spread'] - roll_mean) / roll_std

    # Rolling percentile
    unified[f'spread_pctile_{window}d'] = unified['go_brent_spread'].rolling(window).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100, raw=False
    )

# Spread momentum (rate of change)
for window in [5, 10, 20]:
    unified[f'spread_momentum_{window}d'] = (
        unified['go_brent_spread'] - unified['go_brent_spread'].rolling(window).mean()
    )

print("Current Spread Z-Scores:")
for window in [60, 120, 252]:
    zscore = unified[f'spread_zscore_{window}d'].iloc[-1]
    pctile = unified[f'spread_pctile_{window}d'].iloc[-1]
    print(f"  {window}d z-score: {zscore:.2f}, percentile: {pctile:.1%}")

Current Spread Z-Scores:
  60d z-score: -0.76, percentile: 20.0%
  120d z-score: 0.41, percentile: 56.7%
  252d z-score: 1.05, percentile: 79.4%


In [12]:
# ─── Visualization: Key Derived Series ───

fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

# Panel 1: Prices
ax1 = axes[0]
ax1.plot(unified.index, unified['brent_c1'], color=COLORS['brent'], label='Brent c1', linewidth=0.8)
ax1_twin = ax1.twinx()
ax1_twin.plot(unified.index, unified['gasoil_c1'], color=COLORS['gasoil'], label='Gasoil c1', linewidth=0.8)
ax1.set_ylabel('Brent ($/bbl)', color=COLORS['brent'])
ax1_twin.set_ylabel('Gasoil ($/mt)', color=COLORS['gasoil'])
ax1.set_title('Front Month Prices')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Panel 2: GO-Brent spread with z-score bands
ax2 = axes[1]
ax2.plot(unified.index, unified['go_brent_spread'], color=COLORS['spread'], linewidth=0.8)
ax2.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax2.set_ylabel('$/bbl')
ax2.set_title('GO–Brent Spread ($/bbl equivalent)')

# Panel 3: Spread z-score
ax3 = axes[2]
ax3.plot(unified.index, unified['spread_zscore_60d'], color=COLORS['highlight'], linewidth=0.8, label='60d')
ax3.axhline(y=2, color='red', linestyle='--', alpha=0.5, label='+2σ')
ax3.axhline(y=-2, color='red', linestyle='--', alpha=0.5, label='-2σ')
ax3.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax3.fill_between(unified.index, -2, 2, alpha=0.05, color='green')
ax3.set_ylabel('Z-Score')
ax3.set_title('GO–Brent Spread Z-Score (60d rolling)')
ax3.legend()

# Panel 4: Realized volatility
ax4 = axes[3]
ax4.plot(unified.index, unified['brent_rvol_20d'], color=COLORS['brent'], label='Brent 20d RVol', linewidth=0.8)
ax4.plot(unified.index, unified['gasoil_rvol_20d'], color=COLORS['gasoil'], label='Gasoil 20d RVol', linewidth=0.8)
ax4.set_ylabel('Annualized Vol')
ax4.set_title('20-Day Realized Volatility')
ax4.legend()
ax4.set_xlabel('Date')

plt.tight_layout()
plt.savefig('derived_series_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal derived columns: {len(unified.columns)}")
print(f"Columns: {list(unified.columns)}")


Total derived columns: 76
Columns: ['brent_c1', 'brent_c2', 'brent_c3', 'brent_c4', 'brent_c5', 'brent_c6', 'brent_c7', 'brent_c8', 'brent_c9', 'brent_c10', 'brent_c11', 'brent_c12', 'brent_c13', 'brent_c14', 'brent_c15', 'brent_c16', 'brent_c17', 'gasoil_c1', 'gasoil_c2', 'gasoil_c3', 'gasoil_c4', 'gasoil_c5', 'gasoil_c6', 'gasoil_c7', 'gasoil_c8', 'gasoil_c9', 'gasoil_c10', 'gasoil_c11', 'gasoil_c12', 'gasoil_c13', 'gasoil_c14', 'go_brent_spread', 'go_brent_spread_c2', 'go_brent_spread_c3', 'go_brent_spread_c4', 'go_brent_ratio', 'brent_M1_M2', 'brent_M2_M3', 'brent_M2_M4', 'brent_M1_M6', 'brent_M1_M3', 'brent_M1_M12', 'gasoil_M1_M2', 'gasoil_M2_M3', 'gasoil_M1_M6', 'brent_ret', 'gasoil_ret', 'spread_chg', 'spread_ret_pct', 'brent_ret_5d', 'gasoil_ret_5d', 'spread_chg_5d', 'brent_ret_10d', 'gasoil_ret_10d', 'spread_chg_10d', 'brent_ret_20d', 'gasoil_ret_20d', 'spread_chg_20d', 'brent_rvol_10d', 'gasoil_rvol_10d', 'spread_vol_10d', 'brent_rvol_20d', 'gasoil_rvol_20d', 'spread_vol_20d

### Interpretation — Derived Series

**Trader's Perspective:**
- The GO–Brent spread exhibits clear regime-dependent behavior with distinct high/low volatility periods
- Calendar spreads capture term structure dynamics — backwardation/contango shifts are critical context
- Z-scores relative to 60d and 252d windows provide different signals: short-term for tactical, long-term for structural
- Volatility clustering is evident — high vol begets high vol, informing position sizing

**Next Questions:**
- Are spread widening events clustered in specific vol regimes?
- Does the spread z-score have mean-reverting properties?

---
# ANALYSIS 3: Significant GO–Brent Spread Widening Event Detection

We use multiple statistical thresholds to identify meaningful spread widening events, 
then cluster consecutive detections into discrete events.

In [13]:
# ─── Event Detection Using Multiple Criteria ───

spread = unified['go_brent_spread'].dropna()

# Criterion 1: Absolute z-score threshold (> 1.5σ above mean using full-sample)
mean_spread = spread.mean()
std_spread = spread.std()
threshold_1_5sigma = mean_spread + 1.5 * std_spread
threshold_2sigma = mean_spread + 2.0 * std_spread

# Criterion 2: Rolling 60d z-score > 1.5
rolling_zscore = unified['spread_zscore_60d'].dropna()

# Criterion 3: Rolling 60d percentile > 90th
rolling_pctile = unified['spread_pctile_60d'].dropna()

# Criterion 4: 5-day spread change exceeding 2σ of 5d changes
spread_5d_chg = unified['spread_chg_5d'].dropna()
threshold_5d_move = spread_5d_chg.mean() + 2.0 * spread_5d_chg.std()

print("Event Detection Thresholds:")
print(f"  Full-sample mean spread: ${mean_spread:.2f}/bbl")
print(f"  Full-sample std: ${std_spread:.2f}/bbl")
print(f"  1.5σ threshold: ${threshold_1_5sigma:.2f}/bbl")
print(f"  2.0σ threshold: ${threshold_2sigma:.2f}/bbl")
print(f"  5d change 2σ threshold: ${threshold_5d_move:.2f}/bbl")

# Build event flags
events = pd.DataFrame(index=unified.index)
events['spread'] = unified['go_brent_spread']
events['flag_1_5sigma'] = spread > threshold_1_5sigma
events['flag_2sigma'] = spread > threshold_2sigma
events['flag_zscore_60d'] = unified['spread_zscore_60d'] > 1.5
events['flag_pctile_90'] = unified['spread_pctile_60d'] > 0.90
events['flag_5d_surge'] = unified['spread_chg_5d'] > threshold_5d_move

# Combined: require at least 2 criteria met
events['n_flags'] = events[['flag_1_5sigma', 'flag_zscore_60d', 'flag_pctile_90', 'flag_5d_surge']].sum(axis=1)
events['is_event'] = events['n_flags'] >= 2

print(f"\nDays meeting each criterion:")
for col in ['flag_1_5sigma', 'flag_2sigma', 'flag_zscore_60d', 'flag_pctile_90', 'flag_5d_surge']:
    print(f"  {col}: {events[col].sum()} days ({events[col].mean():.1%})")
print(f"  Combined (≥2 flags): {events['is_event'].sum()} days ({events['is_event'].mean():.1%})")

Event Detection Thresholds:
  Full-sample mean spread: $25.56/bbl
  Full-sample std: $15.35/bbl
  1.5σ threshold: $48.58/bbl
  2.0σ threshold: $56.25/bbl
  5d change 2σ threshold: $11.51/bbl

Days meeting each criterion:
  flag_1_5sigma: 154 days (9.8%)
  flag_2sigma: 82 days (5.2%)
  flag_zscore_60d: 313 days (19.9%)
  flag_pctile_90: 349 days (22.2%)
  flag_5d_surge: 38 days (2.4%)
  Combined (≥2 flags): 318 days (20.3%)


In [14]:
# ─── Cluster Consecutive Event Days into Discrete Events ───

def cluster_events(event_flags, min_gap=5):
    """
    Group consecutive True flags (or flags within min_gap days) into discrete events.
    Returns DataFrame with event_id, start, end, peak_date, peak_value, duration.
    """
    event_dates = event_flags[event_flags].index
    if len(event_dates) == 0:
        return pd.DataFrame()

    events_list = []
    current_event = {'start': event_dates[0], 'dates': [event_dates[0]]}

    for date in event_dates[1:]:
        gap = (date - current_event['dates'][-1]).days
        if gap <= min_gap:
            current_event['dates'].append(date)
        else:
            events_list.append(current_event)
            current_event = {'start': date, 'dates': [date]}
    events_list.append(current_event)

    result = []
    for i, evt in enumerate(events_list):
        start = evt['dates'][0]
        end = evt['dates'][-1]
        duration = (end - start).days + 1

        # Find peak spread during event window
        mask = (unified.index >= start) & (unified.index <= end)
        event_spreads = unified.loc[mask, 'go_brent_spread']
        peak_idx = event_spreads.idxmax() if len(event_spreads) > 0 else start
        peak_val = event_spreads.max() if len(event_spreads) > 0 else np.nan

        # Pre-event spread (5 days before)
        pre_start = start - pd.Timedelta(days=10)
        pre_mask = (unified.index >= pre_start) & (unified.index < start)
        pre_spread = unified.loc[pre_mask, 'go_brent_spread'].mean()

        result.append({
            'event_id': i + 1,
            'start': start,
            'end': end,
            'peak_date': peak_idx,
            'peak_spread': peak_val,
            'pre_event_spread': pre_spread,
            'spread_widening': peak_val - pre_spread if not np.isnan(pre_spread) else np.nan,
            'duration_days': duration,
            'n_event_days': len(evt['dates']),
        })

    return pd.DataFrame(result)

event_db = cluster_events(events['is_event'], min_gap=5)

print(f"\nDETECTED {len(event_db)} DISCRETE SPREAD WIDENING EVENTS")
print("=" * 80)
print(event_db.to_string(index=False))


DETECTED 30 DISCRETE SPREAD WIDENING EVENTS
 event_id      start        end  peak_date  peak_spread  pre_event_spread  spread_widening  duration_days  n_event_days
        1 2021-05-07 2021-05-24 2021-05-20     7.081905          5.006984         2.074921             18            15
        2 2021-08-01 2021-09-07 2021-09-03     9.479299          6.304533         3.174766             38            26
        3 2021-09-13 2021-10-19 2021-10-15    14.876544          8.920918         5.955626             37            30
        4 2022-01-10 2022-02-09 2022-02-04    21.077204         12.717245         8.359959             31            21
        5 2022-03-01 2022-03-09 2022-03-08    79.817488         15.659155        64.158333              9             7
        6 2022-03-18 2022-03-24 2022-03-24    47.234449         32.222648        15.011801              7             6
        7 2022-04-26 2022-05-03 2022-05-02    65.763394         43.876129        21.887265              8          

In [15]:
# ─── Visualize Event Detection ───

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Panel 1: Spread with event highlighting
ax1 = axes[0]
ax1.plot(unified.index, unified['go_brent_spread'], color=COLORS['spread'], linewidth=0.7, alpha=0.8)

# Highlight event windows
for _, evt in event_db.iterrows():
    ax1.axvspan(evt['start'], evt['end'], alpha=0.2, color=COLORS['event'])
    ax1.annotate(f"E{int(evt['event_id'])}", xy=(evt['peak_date'], evt['peak_spread']),
                 fontsize=8, ha='center', va='bottom', color=COLORS['event'], fontweight='bold')

ax1.axhline(y=threshold_1_5sigma, color='orange', linestyle='--', alpha=0.5, label='1.5σ')
ax1.axhline(y=threshold_2sigma, color='red', linestyle='--', alpha=0.5, label='2.0σ')
ax1.axhline(y=mean_spread, color='gray', linestyle=':', alpha=0.5, label='Mean')
ax1.set_ylabel('$/bbl')
ax1.set_title('GO–Brent Spread with Detected Widening Events')
ax1.legend(loc='upper left')

# Panel 2: Z-score with event flags
ax2 = axes[1]
ax2.plot(unified.index, unified['spread_zscore_60d'], color=COLORS['highlight'], linewidth=0.7)
ax2.axhline(y=1.5, color='orange', linestyle='--', alpha=0.5, label='1.5')
ax2.axhline(y=2.0, color='red', linestyle='--', alpha=0.5, label='2.0')
ax2.axhline(y=0, color='gray', linestyle=':', alpha=0.5)

for _, evt in event_db.iterrows():
    ax2.axvspan(evt['start'], evt['end'], alpha=0.2, color=COLORS['event'])

ax2.set_ylabel('Z-Score')
ax2.set_title('60d Rolling Z-Score with Event Windows')
ax2.legend()
ax2.set_xlabel('Date')

plt.tight_layout()
plt.savefig('event_detection.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation — Event Detection

**Trader's Perspective:**
- The multi-criteria approach reduces false positives from any single indicator
- Events vary significantly in duration and magnitude — not all "wide spreads" are created equal
- Clustering prevents double-counting of multi-day widening episodes
- The event database provides a structured framework for systematic back-testing

**Limitations:**
- Thresholds are full-sample based, introducing look-ahead bias for early events
- The 2021+ period is relatively short, limiting sample size
- Minimum gap of 5 days may merge what are genuinely separate episodes

---
# ANALYSIS 4: Historical Propagation Analysis

For each detected widening event, we measure what happened to Brent prices, the GO–Brent 
spread, and calendar spreads at various horizons post-event.

In [16]:
# ─── Propagation Analysis Engine ───

horizons = [1, 2, 3, 5, 7, 10, 15, 20, 30]

def compute_post_event_responses(event_db, data, horizons):
    """
    For each event, compute the response of key variables at each horizon.
    Uses the event peak_date as t=0.
    """
    results = []

    for _, evt in event_db.iterrows():
        t0 = evt['peak_date']
        t0_idx = data.index.get_loc(t0) if t0 in data.index else None
        if t0_idx is None:
            continue

        record = {
            'event_id': evt['event_id'],
            't0_date': t0,
            't0_brent': data.iloc[t0_idx]['brent_c1'] if 'brent_c1' in data.columns else np.nan,
            't0_spread': data.iloc[t0_idx]['go_brent_spread'] if 'go_brent_spread' in data.columns else np.nan,
        }

        for h in horizons:
            future_idx = t0_idx + h
            if future_idx >= len(data):
                for var in ['brent_chg', 'brent_ret', 'spread_chg', 'spread_normalized']:
                    record[f'{var}_t{h}'] = np.nan
                continue

            future = data.iloc[future_idx]
            t0_data = data.iloc[t0_idx]

            # Brent change
            record[f'brent_chg_t{h}'] = future['brent_c1'] - t0_data['brent_c1']
            record[f'brent_ret_t{h}'] = (future['brent_c1'] / t0_data['brent_c1']) - 1

            # Spread change
            if 'go_brent_spread' in data.columns:
                record[f'spread_chg_t{h}'] = future['go_brent_spread'] - t0_data['go_brent_spread']
                # Spread normalization (how much has the spread moved back towards mean)
                initial_deviation = t0_data['go_brent_spread'] - mean_spread
                if abs(initial_deviation) > 0.01:
                    current_deviation = future['go_brent_spread'] - mean_spread
                    record[f'spread_normalized_t{h}'] = 1 - (current_deviation / initial_deviation)
                else:
                    record[f'spread_normalized_t{h}'] = np.nan

            # Calendar spread change
            if 'brent_M1_M2' in data.columns:
                record[f'M1M2_chg_t{h}'] = future['brent_M1_M2'] - t0_data['brent_M1_M2']

        results.append(record)

    return pd.DataFrame(results)

propagation = compute_post_event_responses(event_db, unified, horizons)
print(f"Propagation analysis computed for {len(propagation)} events across {len(horizons)} horizons")
propagation.head()

Propagation analysis computed for 30 events across 9 horizons


,event_id,t0_date,t0_brent,t0_spread,brent_chg_t1,brent_ret_t1,spread_chg_t1,spread_normalized_t1,M1M2_chg_t1,brent_chg_t2,...,brent_chg_t20,brent_ret_t20,spread_chg_t20,spread_normalized_t20,M1M2_chg_t20,brent_chg_t30,brent_ret_t30,spread_chg_t30,spread_normalized_t30,M1M2_chg_t30
0,1,2021-05-20,65.011538,7.081905,1.684712,0.025914,-0.378826,-0.020503,-0.005743,1.450462,...,7.757033,0.119318,-0.634273,-0.034328,0.523801,10.568462,0.162563,-1.472494,-0.079695,0.687293
1,2,2021-09-03,72.526667,9.479299,-0.475000,-0.006549,-0.004120,-0.000256,-0.035462,-0.433333,...,6.850256,0.094452,0.590087,0.036699,0.181590,10.077333,0.138947,3.554285,0.221049,0.087333
2,3,2021-10-15,84.855000,14.876544,0.490000,0.005775,-0.267685,-0.025060,0.034286,-0.720000,...,0.350000,0.004125,-0.176379,-0.016512,0.525000,-2.671667,-0.031485,-4.482276,-0.419612,0.308333
3,4,2022-02-04,92.670000,21.077204,0.314286,0.003391,-0.546389,-0.121927,-0.229464,-1.361111,...,25.332500,0.273362,23.034390,5.140121,2.310000,14.225714,0.153509,11.323982,2.526945,1.355714
4,5,2022-03-08,129.377143,79.817488,-16.841587,-0.130174,-40.227764,0.741403,-0.361587,-20.180476,...,-21.417143,-0.165540,-46.049300,0.848694,-0.863810,-24.462143,-0.189076,-36.645240,0.675376,-4.017143


In [17]:
# ─── Summary Statistics at Each Horizon ───

summary_data = []
for h in horizons:
    brent_col = f'brent_chg_t{h}'
    brent_ret_col = f'brent_ret_t{h}'
    spread_col = f'spread_chg_t{h}'

    if brent_col not in propagation.columns:
        continue

    brent_changes = propagation[brent_col].dropna()
    spread_changes = propagation[spread_col].dropna()

    summary_data.append({
        'Horizon': f't+{h}',
        'N': len(brent_changes),
        'Brent_Avg_Chg': brent_changes.mean(),
        'Brent_Median_Chg': brent_changes.median(),
        'Brent_Std': brent_changes.std(),
        'Brent_Max': brent_changes.max(),
        'Brent_Min': brent_changes.min(),
        'Brent_Up_Pct': (brent_changes > 0).mean(),
        'Spread_Avg_Chg': spread_changes.mean(),
        'Spread_Median_Chg': spread_changes.median(),
        'Spread_Narrows_Pct': (spread_changes < 0).mean(),
    })

summary_df = pd.DataFrame(summary_data)
print("POST-EVENT PROPAGATION SUMMARY")
print("=" * 100)
print(summary_df.to_string(index=False, float_format='{:.3f}'.format))

# Statistical significance: test if mean Brent change is significantly different from zero
print("\n\nSTATISTICAL SIGNIFICANCE (t-test: H0 = mean change = 0)")
print("-" * 70)
for h in horizons:
    brent_col = f'brent_chg_t{h}'
    if brent_col in propagation.columns:
        data_h = propagation[brent_col].dropna()
        if len(data_h) >= 3:
            t_stat, p_val = stats.ttest_1samp(data_h, 0)
            sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.10 else ''
            print(f"  t+{h:2d}: t-stat = {t_stat:6.2f}, p-value = {p_val:.4f} {sig}")

POST-EVENT PROPAGATION SUMMARY
Horizon  N  Brent_Avg_Chg  Brent_Median_Chg  Brent_Std  Brent_Max  Brent_Min  Brent_Up_Pct  Spread_Avg_Chg  Spread_Median_Chg  Spread_Narrows_Pct
    t+1 30         -0.469             0.154      3.289      1.805    -16.842         0.633          -3.910             -1.346               1.000
    t+2 30         -0.495            -0.071      4.410      4.523    -20.180         0.467          -5.569             -3.014               1.000
    t+3 30         -0.349             0.246      5.548     14.030    -16.921         0.567          -7.181             -2.947               1.000
    t+5 30         -0.798            -0.100      6.239      8.156    -23.450         0.433          -8.493             -4.051               0.967
    t+7 30         -1.592             0.106      7.419      7.535    -31.357         0.500          -9.141             -3.911               1.000
   t+10 30         -1.194            -0.506      5.773      5.834    -20.377         0.400   

In [18]:
# ─── Average Historical Path Visualization ───

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Collect paths for each event
brent_paths = {}
spread_paths = {}

for _, evt in propagation.iterrows():
    eid = int(evt['event_id'])
    brent_path = [0]  # t=0 change is 0
    spread_path = [0]

    for h in horizons:
        brent_path.append(evt.get(f'brent_chg_t{h}', np.nan))
        spread_path.append(evt.get(f'spread_chg_t{h}', np.nan))

    brent_paths[eid] = brent_path
    spread_paths[eid] = spread_path

x_axis = [0] + horizons

# Panel 1: Individual Brent paths
ax = axes[0, 0]
for eid, path in brent_paths.items():
    ax.plot(x_axis, path, alpha=0.4, linewidth=0.8, label=f'E{eid}')
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Days After Peak Spread')
ax.set_ylabel('Brent Change ($/bbl)')
ax.set_title('Individual Brent Price Paths Post-Event')
if len(brent_paths) <= 15:
    ax.legend(fontsize=7, ncol=2)

# Panel 2: Average Brent path with confidence band
ax = axes[0, 1]
avg_brent = [np.nanmean([brent_paths[e][i] for e in brent_paths]) for i in range(len(x_axis))]
med_brent = [np.nanmedian([brent_paths[e][i] for e in brent_paths]) for i in range(len(x_axis))]
std_brent = [np.nanstd([brent_paths[e][i] for e in brent_paths]) for i in range(len(x_axis))]

ax.plot(x_axis, avg_brent, color=COLORS['brent'], linewidth=2, label='Mean')
ax.plot(x_axis, med_brent, color=COLORS['brent'], linewidth=2, linestyle='--', label='Median')
ax.fill_between(x_axis, 
                 [a - s for a, s in zip(avg_brent, std_brent)],
                 [a + s for a, s in zip(avg_brent, std_brent)],
                 alpha=0.2, color=COLORS['brent'], label='±1σ')
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Days After Peak Spread')
ax.set_ylabel('Brent Change ($/bbl)')
ax.set_title('Average Brent Path Post-Spread-Widening')
ax.legend()

# Panel 3: Average spread path
ax = axes[1, 0]
avg_spread = [np.nanmean([spread_paths[e][i] for e in spread_paths]) for i in range(len(x_axis))]
med_spread = [np.nanmedian([spread_paths[e][i] for e in spread_paths]) for i in range(len(x_axis))]
std_spread_path = [np.nanstd([spread_paths[e][i] for e in spread_paths]) for i in range(len(x_axis))]

ax.plot(x_axis, avg_spread, color=COLORS['spread'], linewidth=2, label='Mean')
ax.plot(x_axis, med_spread, color=COLORS['spread'], linewidth=2, linestyle='--', label='Median')
ax.fill_between(x_axis,
                 [a - s for a, s in zip(avg_spread, std_spread_path)],
                 [a + s for a, s in zip(avg_spread, std_spread_path)],
                 alpha=0.2, color=COLORS['spread'], label='±1σ')
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Days After Peak Spread')
ax.set_ylabel('Spread Change ($/bbl)')
ax.set_title('Average Spread Path Post-Widening')
ax.legend()

# Panel 4: Hit ratio (probability of Brent up / spread narrowing)
ax = axes[1, 1]
brent_up_pct = []
spread_narrow_pct = []
for h_idx, h in enumerate(horizons):
    brent_vals = [brent_paths[e][h_idx + 1] for e in brent_paths if not np.isnan(brent_paths[e][h_idx + 1])]
    spread_vals = [spread_paths[e][h_idx + 1] for e in spread_paths if not np.isnan(spread_paths[e][h_idx + 1])]
    brent_up_pct.append(np.mean([v > 0 for v in brent_vals]) if brent_vals else np.nan)
    spread_narrow_pct.append(np.mean([v < 0 for v in spread_vals]) if spread_vals else np.nan)

ax.plot(horizons, brent_up_pct, 'o-', color=COLORS['brent'], label='P(Brent ↑)', linewidth=2)
ax.plot(horizons, spread_narrow_pct, 's-', color=COLORS['spread'], label='P(Spread ↓)', linewidth=2)
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='50%')
ax.set_xlabel('Days After Peak Spread')
ax.set_ylabel('Probability')
ax.set_title('Hit Ratio: P(Brent Up) & P(Spread Narrows)')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('propagation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [19]:
# ─── Time to Peak and Normalization ───

print("EVENT-LEVEL DETAIL")
print("=" * 80)

for _, evt in propagation.iterrows():
    eid = int(evt['event_id'])
    # Find peak Brent increase
    max_brent_chg = -np.inf
    peak_h = 0
    for h in horizons:
        chg = evt.get(f'brent_chg_t{h}', np.nan)
        if not np.isnan(chg) and chg > max_brent_chg:
            max_brent_chg = chg
            peak_h = h

    # Find time to spread normalization (first h where spread_chg < 0)
    norm_h = None
    for h in horizons:
        chg = evt.get(f'spread_chg_t{h}', np.nan)
        if not np.isnan(chg) and chg < 0:
            norm_h = h
            break

    print(f"Event {eid} ({evt['t0_date'].date()}):")
    print(f"  Initial spread: ${evt['t0_spread']:.2f}/bbl")
    print(f"  Peak Brent move: +${max_brent_chg:.2f} at t+{peak_h}")
    if norm_h:
        print(f"  Spread starts narrowing: t+{norm_h}")
    else:
        print(f"  Spread did NOT narrow within 30 days")
    print()

EVENT-LEVEL DETAIL
Event 1 (2021-05-20):
  Initial spread: $7.08/bbl
  Peak Brent move: +$10.57 at t+30
  Spread starts narrowing: t+1

Event 2 (2021-09-03):
  Initial spread: $9.48/bbl
  Peak Brent move: +$10.08 at t+30
  Spread starts narrowing: t+1

Event 3 (2021-10-15):
  Initial spread: $14.88/bbl
  Peak Brent move: +$0.75 at t+7
  Spread starts narrowing: t+1

Event 4 (2022-02-04):
  Initial spread: $21.08/bbl
  Peak Brent move: +$25.33 at t+20
  Spread starts narrowing: t+1

Event 5 (2022-03-08):
  Initial spread: $79.82/bbl
  Peak Brent move: +$-9.64 at t+15
  Spread starts narrowing: t+1

Event 6 (2022-03-24):
  Initial spread: $47.23/bbl
  Peak Brent move: +$1.52 at t+1
  Spread starts narrowing: t+1

Event 7 (2022-05-02):
  Initial spread: $65.76/bbl
  Peak Brent move: +$12.42 at t+30
  Spread starts narrowing: t+1

Event 8 (2022-06-22):
  Initial spread: $63.34/bbl
  Peak Brent move: +$8.16 at t+5
  Spread starts narrowing: t+1

Event 9 (2022-07-12):
  Initial spread: $55.2

### Interpretation — Historical Propagation

**Trader's Perspective:**
- The average Brent path and spread path reveal the typical reaction profile
- Hit ratios above 50% suggest a directional edge; below 50% suggests noise
- The time-to-peak Brent move indicates how quickly the market absorbs the information
- Wide confidence bands indicate significant dispersion — any individual event outcome is uncertain
- t-test results quantify whether the average move is statistically distinguishable from randomness

**Key Metric:** If P(Brent ↑) at t+5 or t+10 is consistently above 55-60%, this could be tradeable

---
# ANALYSIS 5: Lead-Lag Relationship Discovery

We systematically test for lead-lag relationships using cross-correlation, Granger causality, 
VAR models, and impulse response functions. We do NOT assume any particular direction of causality.

In [20]:
# ─── Cross-Correlation Analysis ───

# Test returns cross-correlation at various lags
max_lag = 20

pairs = [
    ('spread_chg', 'brent_ret', 'Spread Δ → Brent Return'),
    ('brent_ret', 'spread_chg', 'Brent Return → Spread Δ'),
    ('gasoil_ret', 'brent_ret', 'Gasoil Return → Brent Return'),
    ('brent_ret', 'gasoil_ret', 'Brent Return → Gasoil Return'),
    ('spread_chg', 'gasoil_ret', 'Spread Δ → Gasoil Return'),
]

fig, axes = plt.subplots(len(pairs), 1, figsize=(14, 4 * len(pairs)))

cross_corr_results = []

for idx, (x_var, y_var, title) in enumerate(pairs):
    x = unified[x_var].dropna()
    y = unified[y_var].dropna()

    # Align
    common = x.index.intersection(y.index)
    x = x.loc[common]
    y = y.loc[common]

    lags = range(-max_lag, max_lag + 1)
    correlations = []

    for lag in lags:
        if lag >= 0:
            corr = x.iloc[:len(x)-lag].corr(y.iloc[lag:].reset_index(drop=True))
        else:
            corr = x.iloc[-lag:].reset_index(drop=True).corr(y.iloc[:len(y)+lag])
        correlations.append(corr)

    # Find peak correlation
    abs_corr = [abs(c) if not np.isnan(c) else 0 for c in correlations]
    peak_idx = np.argmax(abs_corr)
    peak_lag = list(lags)[peak_idx]
    peak_corr = correlations[peak_idx]

    # Significance threshold (approximate)
    n = len(common)
    sig_threshold = 2 / np.sqrt(n)

    cross_corr_results.append({
        'Pair': title,
        'Peak_Lag': peak_lag,
        'Peak_Corr': peak_corr,
        'Significant': abs(peak_corr) > sig_threshold,
    })

    # Plot
    ax = axes[idx]
    colors = ['red' if abs(c) > sig_threshold else 'steelblue' for c in correlations]
    ax.bar(list(lags), correlations, color=colors, alpha=0.7)
    ax.axhline(y=sig_threshold, color='red', linestyle='--', alpha=0.5)
    ax.axhline(y=-sig_threshold, color='red', linestyle='--', alpha=0.5)
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
    ax.set_title(f'{title} (Peak: lag={peak_lag}, r={peak_corr:.3f})')
    ax.set_xlabel('Lag (days)')
    ax.set_ylabel('Correlation')

plt.tight_layout()
plt.savefig('cross_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCross-Correlation Summary:")
print(pd.DataFrame(cross_corr_results).to_string(index=False))


Cross-Correlation Summary:
                        Pair  Peak_Lag  Peak_Corr  Significant
     Spread Δ → Brent Return       -20        NaN        False
     Brent Return → Spread Δ       -20        NaN        False
Gasoil Return → Brent Return       -20        NaN        False
Brent Return → Gasoil Return       -20        NaN        False
    Spread Δ → Gasoil Return       -20        NaN        False


In [21]:
# ─── Granger Causality Tests ───
# Test both directions: Spread → Brent and Brent → Spread

granger_vars = unified[['brent_ret', 'gasoil_ret', 'spread_chg']].dropna()

print("GRANGER CAUSALITY TESTS")
print("=" * 80)
print("H0: X does NOT Granger-cause Y")
print("Small p-values suggest X helps predict Y beyond its own history")
print()

test_pairs = [
    ('spread_chg', 'brent_ret', 'Spread change → Brent return'),
    ('brent_ret', 'spread_chg', 'Brent return → Spread change'),
    ('gasoil_ret', 'brent_ret', 'Gasoil return → Brent return'),
    ('brent_ret', 'gasoil_ret', 'Brent return → Gasoil return'),
    ('spread_chg', 'gasoil_ret', 'Spread change → Gasoil return'),
    ('gasoil_ret', 'spread_chg', 'Gasoil return → Spread change'),
]

granger_results = []
max_gc_lag = 10

for x_col, y_col, description in test_pairs:
    test_data = granger_vars[[y_col, x_col]].dropna()

    if len(test_data) < 50:
        continue

    print(f"\n{description}:")
    try:
        gc_result = grangercausalitytests(test_data, maxlag=max_gc_lag, verbose=False)

        for lag in [1, 2, 3, 5, 10]:
            if lag <= max_gc_lag:
                f_stat = gc_result[lag][0]['ssr_ftest'][0]
                p_val = gc_result[lag][0]['ssr_ftest'][1]
                sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.10 else ''
                print(f"  Lag {lag:2d}: F={f_stat:7.3f}, p={p_val:.4f} {sig}")

                granger_results.append({
                    'Direction': description,
                    'Lag': lag,
                    'F_stat': f_stat,
                    'P_value': p_val,
                    'Significant_5pct': p_val < 0.05,
                })
    except Exception as e:
        print(f"  Error: {e}")

gc_df = pd.DataFrame(granger_results)
print("\n\nGRANGER CAUSALITY SIGNIFICANT RESULTS (p < 0.05):")
significant_gc = gc_df[gc_df['Significant_5pct']].sort_values('P_value')
if len(significant_gc) > 0:
    print(significant_gc.to_string(index=False))
else:
    print("  No significant Granger causality found at 5% level")

GRANGER CAUSALITY TESTS
H0: X does NOT Granger-cause Y
Small p-values suggest X helps predict Y beyond its own history


Spread change → Brent return:
  Lag  1: F=  7.779, p=0.0053 ***
  Lag  2: F=  7.624, p=0.0005 ***
  Lag  3: F=  5.156, p=0.0015 ***
  Lag  5: F=  3.125, p=0.0082 ***
  Lag 10: F=  3.209, p=0.0004 ***

Brent return → Spread change:
  Lag  1: F=  1.377, p=0.2409 
  Lag  2: F=  1.599, p=0.2025 
  Lag  3: F=  1.108, p=0.3446 
  Lag  5: F=  1.806, p=0.1086 
  Lag 10: F=  1.650, p=0.0874 *

Gasoil return → Brent return:
  Lag  1: F=  9.161, p=0.0025 ***
  Lag  2: F=  9.838, p=0.0001 ***
  Lag  3: F=  7.071, p=0.0001 ***
  Lag  5: F=  4.289, p=0.0007 ***
  Lag 10: F=  3.294, p=0.0003 ***

Brent return → Gasoil return:
  Lag  1: F=  1.571, p=0.2103 
  Lag  2: F=  3.579, p=0.0281 **
  Lag  3: F=  2.534, p=0.0554 *
  Lag  5: F=  2.051, p=0.0689 *
  Lag 10: F=  2.019, p=0.0282 **

Spread change → Gasoil return:
  Lag  1: F=  0.548, p=0.4594 
  Lag  2: F=  0.369, p=0.6914 
  Lag

  Lag  1: F=  2.991, p=0.0839 *
  Lag  2: F=  1.792, p=0.1670 
  Lag  3: F=  1.496, p=0.2138 
  Lag  5: F=  1.898, p=0.0917 *
  Lag 10: F=  1.799, p=0.0561 *


GRANGER CAUSALITY SIGNIFICANT RESULTS (p < 0.05):
                   Direction  Lag   F_stat  P_value  Significant_5pct
Gasoil return → Brent return    2 9.838090 0.000057              True
Gasoil return → Brent return    3 7.070701 0.000102              True
Gasoil return → Brent return   10 3.294098 0.000309              True
Spread change → Brent return   10 3.209296 0.000425              True
Spread change → Brent return    2 7.623601 0.000507              True
Gasoil return → Brent return    5 4.289065 0.000705              True
Spread change → Brent return    3 5.155503 0.001508              True
Gasoil return → Brent return    1 9.161036 0.002513              True
Spread change → Brent return    1 7.778971 0.005350              True
Spread change → Brent return    5 3.125184 0.008210              True
Brent return → Gasoi

In [22]:
# ─── Rolling Correlation & Beta ───

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for window, alpha in [(30, 0.4), (60, 0.7), (120, 1.0)]:
    # Rolling correlation: Brent vs Gasoil returns
    roll_corr = unified['brent_ret'].rolling(window).corr(unified['gasoil_ret'])
    axes[0].plot(unified.index, roll_corr, alpha=alpha, label=f'{window}d', linewidth=0.8)

    # Rolling correlation: Spread change vs Brent return
    roll_corr_sb = unified['spread_chg'].rolling(window).corr(unified['brent_ret'])
    axes[1].plot(unified.index, roll_corr_sb, alpha=alpha, label=f'{window}d', linewidth=0.8)

axes[0].axhline(y=0, color='gray', linestyle=':', alpha=0.5)
axes[0].set_title('Rolling Correlation: Brent vs Gasoil Returns')
axes[0].set_ylabel('Correlation')
axes[0].legend()

axes[1].axhline(y=0, color='gray', linestyle=':', alpha=0.5)
axes[1].set_title('Rolling Correlation: Spread Change vs Brent Return')
axes[1].set_ylabel('Correlation')
axes[1].legend()

# Rolling beta: Gasoil return regressed on Brent return
for window in [60, 120]:
    roll_cov = unified['gasoil_ret'].rolling(window).cov(unified['brent_ret'])
    roll_var = unified['brent_ret'].rolling(window).var()
    roll_beta = roll_cov / roll_var
    axes[2].plot(unified.index, roll_beta, label=f'β(GO|Brent) {window}d', linewidth=0.8)

axes[2].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
axes[2].set_title('Rolling Beta: Gasoil on Brent')
axes[2].set_ylabel('Beta')
axes[2].legend()
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.savefig('rolling_correlation_beta.png', dpi=150, bbox_inches='tight')
plt.show()

In [23]:
# ─── VAR Model & Impulse Response Functions ───

# Prepare stationary series
var_data = unified[['brent_ret', 'gasoil_ret', 'spread_chg']].dropna()

# Check stationarity
print("STATIONARITY TESTS (Augmented Dickey-Fuller)")
print("-" * 50)
for col in var_data.columns:
    adf_stat, p_val, _, _, crit, _ = adfuller(var_data[col].dropna())
    status = 'STATIONARY' if p_val < 0.05 else 'NON-STATIONARY'
    print(f"  {col:20s}: ADF={adf_stat:.3f}, p={p_val:.4f} → {status}")

# Fit VAR model
print("\n\nVAR MODEL ESTIMATION")
print("=" * 60)

# Select optimal lag
model = VAR(var_data)
lag_order = model.select_order(maxlags=15)
print(f"Optimal lag selection:")
print(lag_order.summary())

# Fit with BIC-selected lag (minimum 1 lag for IRF analysis)
optimal_lag = max(lag_order.bic, 1)
var_result = model.fit(optimal_lag)
print(f"\nFitted VAR({optimal_lag}) [BIC suggested {lag_order.bic}, using max(bic, 1)]")
print(var_result.summary())

STATIONARITY TESTS (Augmented Dickey-Fuller)
--------------------------------------------------
  brent_ret           : ADF=-8.440, p=0.0000 → STATIONARY
  gasoil_ret          : ADF=-16.916, p=0.0000 → STATIONARY
  spread_chg          : ADF=-10.658, p=0.0000 → STATIONARY


VAR MODEL ESTIMATION


Optimal lag selection:
 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -16.27     -16.26*   8.555e-08      -16.27
1       -16.29      -16.25   8.450e-08      -16.27
2       -16.30      -16.23   8.299e-08      -16.28
3       -16.33      -16.23   8.102e-08     -16.29*
4       -16.33      -16.20   8.083e-08      -16.28
5       -16.33      -16.16   8.096e-08      -16.27
6       -16.34      -16.15   7.987e-08      -16.27
7       -16.34      -16.11   8.005e-08      -16.26
8       -16.34      -16.08   8.009e-08      -16.24
9       -16.34      -16.06   7.971e-08      -16.24
10     -16.35*      -16.03  7.951e-08*      -16.23
11      -16.34      -15.99   7.993e-08      -16.21
12      -16.34      -15.96   8.023e-08      -16.20
13      -16.33      -15.92   8.088e-08      -16.18
14      -16.34      -15.89   8.026e-08      -16.17
15      -16.33      -15.86   8.057e-08      -16.16
--------

C:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [24]:
# ─── Impulse Response Functions ───

try:
    irf = var_result.irf(periods=30)

    fig, axes = plt.subplots(3, 3, figsize=(16, 14))

    var_names = ['brent_ret', 'gasoil_ret', 'spread_chg']
    var_labels = ['Brent Return', 'Gasoil Return', 'Spread Change']

    for i, (resp_var, resp_label) in enumerate(zip(var_names, var_labels)):
        for j, (shock_var, shock_label) in enumerate(zip(var_names, var_labels)):
            ax = axes[i, j]

            # Get IRF values
            irf_vals = irf.irfs[:, i, j]
            periods_range = range(len(irf_vals))
            ax.plot(periods_range, irf_vals, color=COLORS['brent'], linewidth=1.5)

            # Confidence intervals (may not always be available)
            try:
                lower = irf.ci[:, i, j, 0]
                upper = irf.ci[:, i, j, 1]
                ax.fill_between(periods_range, lower, upper, alpha=0.2, color=COLORS['brent'])
            except (AttributeError, IndexError):
                pass

            ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
            ax.set_title(f'{shock_label} → {resp_label}', fontsize=10)

            if i == 2:
                ax.set_xlabel('Days')
            if j == 0:
                ax.set_ylabel('Response')

    plt.suptitle('Impulse Response Functions (VAR Model)', fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('impulse_response_functions.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Print key IRF findings
    print("IRF SUMMARY (cumulative response at key horizons):")
    print("-" * 60)
    for i, resp_label in enumerate(var_labels):
        for j, shock_label in enumerate(var_labels):
            if i != j:
                resp_5 = irf.irfs[min(5, len(irf.irfs)-1), i, j]
                resp_10 = irf.irfs[min(10, len(irf.irfs)-1), i, j]
                print(f"  {shock_label:20s} → {resp_label:20s}: 5d={resp_5:.6f}, 10d={resp_10:.6f}")
except Exception as e:
    print(f"IRF computation failed: {e}")
    print("This can occur when the optimal VAR lag is very low. Proceeding with remaining analyses.")

IRF SUMMARY (cumulative response at key horizons):
------------------------------------------------------------
  Gasoil Return        → Brent Return        : 5d=-0.000003, 10d=0.000000
  Spread Change        → Brent Return        : 5d=0.000000, 10d=-0.000000
  Brent Return         → Gasoil Return       : 5d=-0.000010, 10d=0.000000
  Spread Change        → Gasoil Return       : 5d=-0.000000, 10d=-0.000000
  Brent Return         → Spread Change       : 5d=-0.002996, 10d=0.000000
  Gasoil Return        → Spread Change       : 5d=0.003161, 10d=-0.000000


In [25]:
# ─── Cointegration Tests ───
# Test whether Brent and Gasoil prices are cointegrated (long-run equilibrium)

print("COINTEGRATION TESTS")
print("=" * 60)

# Engle-Granger test on price levels
brent_prices = unified['brent_c1'].dropna()
gasoil_prices = unified['gasoil_c1'].dropna()
common = brent_prices.index.intersection(gasoil_prices.index)

t_stat, p_val, crit_vals = coint(brent_prices.loc[common], gasoil_prices.loc[common])
print(f"\nEngle-Granger Cointegration Test (Brent ~ Gasoil):")
print(f"  Test statistic: {t_stat:.4f}")
print(f"  P-value: {p_val:.4f}")
print(f"  Critical values: 1%={crit_vals[0]:.4f}, 5%={crit_vals[1]:.4f}, 10%={crit_vals[2]:.4f}")
print(f"  Result: {'COINTEGRATED' if p_val < 0.05 else 'NOT cointegrated'} at 5% level")

# Johansen test on the full system
from statsmodels.tsa.vector_ar.vecm import coint_johansen

price_data = unified[['brent_c1', 'gasoil_c1']].dropna()
johansen_result = coint_johansen(price_data, det_order=0, k_ar_diff=2)

print(f"\nJohansen Cointegration Test:")
print(f"  Trace statistics: {johansen_result.lr1}")
print(f"  Critical values (95%): {johansen_result.cvt[:, 1]}")
print(f"  Eigenvalues: {johansen_result.eig}")

n_coint = sum(johansen_result.lr1 > johansen_result.cvt[:, 1])
print(f"  Number of cointegrating relationships: {n_coint}")

if n_coint > 0:
    print(f"  → Brent and Gasoil share a long-run equilibrium relationship")
    print(f"  → The GO-Brent spread is mean-reverting in the long run")
else:
    print(f"  → No evidence of long-run equilibrium between Brent and Gasoil")

COINTEGRATION TESTS

Engle-Granger Cointegration Test (Brent ~ Gasoil):
  Test statistic: -2.6204
  P-value: 0.2295
  Critical values: 1%=-3.9034, 5%=-3.3400, 10%=-3.0472
  Result: NOT cointegrated at 5% level

Johansen Cointegration Test:
  Trace statistics: [28.0130674   6.70708831]
  Critical values (95%): [15.4943  3.8415]
  Eigenvalues: [0.01350465 0.00427106]
  Number of cointegrating relationships: 2
  → Brent and Gasoil share a long-run equilibrium relationship
  → The GO-Brent spread is mean-reverting in the long run


In [26]:
# ─── Lead-Lag Matrix: Rank All Relationships ───

print("\nLEAD-LAG RELATIONSHIP RANKING")
print("=" * 80)
print("Ranked by statistical significance and predictive power")
print()

# Combine cross-correlation and Granger results
all_relationships = []

for _, row in pd.DataFrame(cross_corr_results).iterrows():
    all_relationships.append({
        'Relationship': row['Pair'],
        'Method': 'Cross-Correlation',
        'Best_Lag': row['Peak_Lag'],
        'Statistic': abs(row['Peak_Corr']),
        'Direction': 'Positive' if row['Peak_Corr'] > 0 else 'Negative',
        'Significant': row['Significant'],
    })

for _, row in gc_df.iterrows():
    if row['Significant_5pct']:
        all_relationships.append({
            'Relationship': row['Direction'],
            'Method': 'Granger Causality',
            'Best_Lag': row['Lag'],
            'Statistic': row['F_stat'],
            'Direction': 'Causal',
            'Significant': True,
        })

rank_df = pd.DataFrame(all_relationships)
rank_df = rank_df.sort_values('Statistic', ascending=False)
print(rank_df.to_string(index=False))


LEAD-LAG RELATIONSHIP RANKING
Ranked by statistical significance and predictive power

                Relationship            Method  Best_Lag  Statistic Direction  Significant
Gasoil return → Brent return Granger Causality         2   9.838090    Causal         True
Gasoil return → Brent return Granger Causality         1   9.161036    Causal         True
Spread change → Brent return Granger Causality         1   7.778971    Causal         True
Spread change → Brent return Granger Causality         2   7.623601    Causal         True
Gasoil return → Brent return Granger Causality         3   7.070701    Causal         True
Spread change → Brent return Granger Causality         3   5.155503    Causal         True
Gasoil return → Brent return Granger Causality         5   4.289065    Causal         True
Brent return → Gasoil return Granger Causality         2   3.579291    Causal         True
Gasoil return → Brent return Granger Causality        10   3.294098    Causal         True
Sp

### Interpretation — Lead-Lag Relationships

**Trader's Perspective:**
- Cross-correlations reveal the raw contemporaneous and lagged relationships between variables
- Granger causality tests whether past values of one variable help predict another BEYOND its own history
- The VAR impulse responses show how a shock to one variable propagates through the system over time
- Cointegration determines whether Brent and Gasoil share a long-run equilibrium (mean-reverting spread)
- The ranking table provides a prioritized view of which relationships are most statistically robust

**Critical Caveat:** Statistical "causality" ≠ economic causality. These tests measure predictive power, not mechanism.

---
# ANALYSIS 6: Feedback Loop Quantification

We test the specific hypothesized feedback mechanism: spread widens → Brent rises → spread narrows → repeat.
If the data supports this loop, we quantify its parameters. If not, we state so clearly.

In [27]:
# ─── Test the Feedback Loop Hypothesis ───
# The proposed loop: Spread widens → Brent rises → Spread narrows
# We need to test each leg independently

print("FEEDBACK LOOP HYPOTHESIS TESTING")
print("=" * 80)

# LEG 1: Spread widens → Brent rises (over following N days)
print("\nLEG 1: After spread widens, does Brent rise?")
print("-" * 60)

# Use rolling windows to avoid event-specific analysis
# Define "wide" days as spread_zscore_60d > 1.0
wide_days = unified[unified['spread_zscore_60d'] > 1.0].index
normal_days = unified[(unified['spread_zscore_60d'] > -0.5) & (unified['spread_zscore_60d'] < 0.5)].index

for horizon in [1, 3, 5, 10, 20]:
    col = f'brent_ret_{horizon}d' if horizon > 1 else 'brent_ret'
    if horizon == 1:
        # Forward return
        forward_ret = unified['brent_c1'].pct_change(1).shift(-1)
    else:
        forward_ret = unified['brent_c1'].pct_change(horizon).shift(-horizon)

    wide_rets = forward_ret.loc[forward_ret.index.isin(wide_days)].dropna()
    normal_rets = forward_ret.loc[forward_ret.index.isin(normal_days)].dropna()

    if len(wide_rets) < 10 or len(normal_rets) < 10:
        continue

    # Two-sample t-test
    t_stat, p_val = stats.ttest_ind(wide_rets, normal_rets)

    print(f"  {horizon}d forward: Wide spread avg={wide_rets.mean():.4f}, "
          f"Normal avg={normal_rets.mean():.4f}, diff={wide_rets.mean()-normal_rets.mean():.4f}, "
          f"t={t_stat:.2f}, p={p_val:.4f} {'**' if p_val < 0.05 else '*' if p_val < 0.10 else ''}")

# LEG 2: After Brent rises, does spread narrow?
print("\nLEG 2: After Brent rises significantly, does spread narrow?")
print("-" * 60)

brent_up_days = unified[unified['brent_ret'] > unified['brent_ret'].quantile(0.8)].index
brent_down_days = unified[unified['brent_ret'] < unified['brent_ret'].quantile(0.2)].index

for horizon in [1, 3, 5, 10, 20]:
    forward_spread_chg = (unified['go_brent_spread'].shift(-horizon) - unified['go_brent_spread'])

    up_spread = forward_spread_chg.loc[forward_spread_chg.index.isin(brent_up_days)].dropna()
    down_spread = forward_spread_chg.loc[forward_spread_chg.index.isin(brent_down_days)].dropna()

    if len(up_spread) < 10:
        continue

    # Test if spread narrows (negative change) after Brent up days
    t_stat, p_val = stats.ttest_1samp(up_spread, 0)

    print(f"  {horizon}d: After Brent up, spread chg avg={up_spread.mean():.4f}, "
          f"t={t_stat:.2f}, p={p_val:.4f} {'**' if p_val < 0.05 else '*' if p_val < 0.10 else ''} "
          f"[narrows: {(up_spread < 0).mean():.1%}]")

FEEDBACK LOOP HYPOTHESIS TESTING

LEG 1: After spread widens, does Brent rise?
------------------------------------------------------------
  1d forward: Wide spread avg=0.0029, Normal avg=0.0007, diff=0.0021, t=1.24, p=0.2153 
  3d forward: Wide spread avg=0.0069, Normal avg=0.0033, diff=0.0036, t=1.27, p=0.2062 
  5d forward: Wide spread avg=0.0078, Normal avg=0.0071, diff=0.0007, t=0.21, p=0.8342 
  10d forward: Wide spread avg=0.0122, Normal avg=0.0140, diff=-0.0018, t=-0.38, p=0.7077 
  20d forward: Wide spread avg=0.0263, Normal avg=0.0222, diff=0.0040, t=0.61, p=0.5406 

LEG 2: After Brent rises significantly, does spread narrow?
------------------------------------------------------------
  1d: After Brent up, spread chg avg=0.2346, t=0.99, p=0.3225  [narrows: 45.9%]
  3d: After Brent up, spread chg avg=0.2138, t=0.57, p=0.5667  [narrows: 43.6%]
  5d: After Brent up, spread chg avg=0.4165, t=0.92, p=0.3558  [narrows: 46.0%]
  10d: After Brent up, spread chg avg=0.4614, t=0.96, 

In [28]:
# ─── Full Loop Timing Analysis ───

print("\nFULL LOOP ANALYSIS")
print("=" * 80)

# For each event: measure
# 1. Time from peak spread to peak Brent (Leg 1 delay)
# 2. Time from peak Brent to spread starting to narrow (Leg 2 delay)
# 3. Time for spread to return to pre-event levels (Leg 3: normalization)

loop_stats = []

for _, evt in propagation.iterrows():
    eid = int(evt['event_id'])
    t0 = evt['t0_date']
    t0_idx = unified.index.get_loc(t0) if t0 in unified.index else None
    if t0_idx is None:
        continue

    # Window: look forward 40 trading days
    window = min(40, len(unified) - t0_idx - 1)
    if window < 10:
        continue

    future = unified.iloc[t0_idx:t0_idx + window + 1]

    # Leg 1: Time to peak Brent
    brent_cummax_idx = (future['brent_c1'] - future['brent_c1'].iloc[0]).idxmax()
    days_to_peak_brent = (brent_cummax_idx - t0).days
    peak_brent_chg = future['brent_c1'].loc[brent_cummax_idx] - future['brent_c1'].iloc[0]

    # Leg 2: After Brent peaks, time for spread to start narrowing
    spread_at_brent_peak = future['go_brent_spread'].loc[brent_cummax_idx] if brent_cummax_idx in future.index else np.nan

    # Leg 3: Time to spread normalization (return below mean + 0.5σ)
    norm_threshold = mean_spread + 0.5 * std_spread
    normalized_dates = future[future['go_brent_spread'] < norm_threshold].index
    if len(normalized_dates) > 0:
        first_norm = normalized_dates[0]
        days_to_norm = (first_norm - t0).days
    else:
        days_to_norm = np.nan

    loop_stats.append({
        'Event': eid,
        'Date': t0.date(),
        'Initial_Spread': evt['t0_spread'],
        'Days_to_Peak_Brent': days_to_peak_brent,
        'Peak_Brent_Chg': peak_brent_chg,
        'Days_to_Normalization': days_to_norm,
    })

loop_df = pd.DataFrame(loop_stats)
print(loop_df.to_string(index=False))

print("\n\nFEEDBACK LOOP SUMMARY:")
print("-" * 50)
if len(loop_df) > 0:
    print(f"Average delay to peak Brent: {loop_df['Days_to_Peak_Brent'].mean():.1f} days (median: {loop_df['Days_to_Peak_Brent'].median():.1f})")
    print(f"Average peak Brent change: ${loop_df['Peak_Brent_Chg'].mean():.2f}/bbl")
    norm_times = loop_df['Days_to_Normalization'].dropna()
    if len(norm_times) > 0:
        print(f"Average normalization time: {norm_times.mean():.1f} days (median: {norm_times.median():.1f})")
        print(f"Events that normalized within 30d: {(norm_times <= 30).sum()}/{len(loop_df)} ({(norm_times <= 30).mean():.0%})")
    else:
        print("No events normalized within the analysis window")

# Confidence interval on delay
if len(loop_df) >= 3:
    ci_delay = stats.t.interval(0.95, len(loop_df)-1,
                                 loc=loop_df['Days_to_Peak_Brent'].mean(),
                                 scale=stats.sem(loop_df['Days_to_Peak_Brent']))
    print(f"\n95% CI for delay to peak Brent: [{ci_delay[0]:.1f}, {ci_delay[1]:.1f}] days")


FULL LOOP ANALYSIS
 Event       Date  Initial_Spread  Days_to_Peak_Brent  Peak_Brent_Chg  Days_to_Normalization
     1 2021-05-20        7.081905                  46       12.091319                    0.0
     2 2021-09-03        9.479299                  47       13.355686                    0.0
     3 2021-10-15       14.876544                  11        1.293462                    0.0
     4 2022-02-04       21.077204                  32       36.707143                    0.0
     5 2022-03-08       79.817488                   0        0.000000                    2.0
     6 2022-03-24       47.234449                   1        1.521429                    5.0
     7 2022-05-02       65.763394                  37       16.282857                   14.0
     8 2022-06-22       63.338819                   6        8.156000                   47.0
     9 2022-07-12       55.256309                  17       10.763333                   27.0
    10 2022-09-07       57.308196                 

### Interpretation — Feedback Loop

**Trader's Perspective:**
- The analysis quantifies whether the proposed feedback mechanism has historical support
- If Leg 1 (spread → Brent) is statistically significant but Leg 2 (Brent → spread narrowing) is not, 
  the "loop" is really a one-directional relationship
- The normalization time indicates how quickly the spread reverts — critical for trade holding period
- Wide confidence intervals on timing parameters suggest the relationship, if it exists, is not mechanically precise

**Verdict:** The statistical evidence for each leg is reported above. A true feedback loop requires 
both legs to be significant with reasonable timing consistency.

---
# ANALYSIS 7: Market Regime Analysis

We identify distinct market regimes automatically and examine how the GO–Brent relationship 
varies across regimes.

In [29]:
# ─── Regime Detection Using Gaussian Mixture Models ───

# Features for regime classification
regime_features = unified[['brent_rvol_20d', 'brent_M1_M2', 'go_brent_spread']].dropna()

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(regime_features)

# Select optimal number of regimes using BIC
bic_scores = []
for n in range(2, 7):
    gmm = GaussianMixture(n_components=n, covariance_type='full', random_state=42, n_init=5)
    gmm.fit(X_scaled)
    bic_scores.append({'n_regimes': n, 'BIC': gmm.bic(X_scaled), 'AIC': gmm.aic(X_scaled)})

bic_df = pd.DataFrame(bic_scores)
optimal_n = bic_df.loc[bic_df['BIC'].idxmin(), 'n_regimes']
print(f"BIC Scores: {bic_df.to_string(index=False)}")
print(f"Optimal regimes: {int(optimal_n)}")

# Fit final model
gmm_final = GaussianMixture(n_components=int(optimal_n), covariance_type='full', random_state=42, n_init=10)
regime_labels = gmm_final.fit_predict(X_scaled)
regime_probs = gmm_final.predict_proba(X_scaled)

regime_features['regime'] = regime_labels
unified.loc[regime_features.index, 'regime'] = regime_labels

BIC Scores:  n_regimes         BIC         AIC
         2 8751.104509 8649.530315
         3 8378.024658 8222.990362
         4 8196.907890 7988.413492
         5 8242.302735 7980.348235
         6 8172.622821 7857.208218
Optimal regimes: 6


In [30]:
# ─── Characterize Each Regime ───

regime_chars = []
regime_names = {}

for r in sorted(regime_features['regime'].unique()):
    mask = regime_features['regime'] == r
    subset = regime_features[mask]

    avg_vol = subset['brent_rvol_20d'].mean()
    avg_cal_spread = subset['brent_M1_M2'].mean()
    avg_go_spread = subset['go_brent_spread'].mean()

    # Name the regime based on characteristics
    if avg_vol > 0.4:
        vol_label = 'High Vol'
    elif avg_vol > 0.25:
        vol_label = 'Med Vol'
    else:
        vol_label = 'Low Vol'

    if avg_cal_spread > 0.5:
        structure_label = 'Backwardation'
    elif avg_cal_spread < -0.5:
        structure_label = 'Contango'
    else:
        structure_label = 'Flat'

    name = f"{vol_label} / {structure_label}"
    regime_names[r] = name

    regime_chars.append({
        'Regime': r,
        'Name': name,
        'Days': mask.sum(),
        'Pct': f'{mask.mean():.1%}',
        'Avg_Vol': f'{avg_vol:.2f}',
        'Avg_M1M2': f'{avg_cal_spread:.2f}',
        'Avg_GO_Spread': f'{avg_go_spread:.2f}',
    })

print("REGIME CHARACTERIZATION")
print("=" * 80)
print(pd.DataFrame(regime_chars).to_string(index=False))

# Map regime names
unified['regime_name'] = unified['regime'].map(regime_names)

REGIME CHARACTERIZATION
 Regime                     Name  Days   Pct Avg_Vol Avg_M1M2 Avg_GO_Spread
      0  Med Vol / Backwardation   214 13.8%    0.35     0.65         11.90
      1 High Vol / Backwardation   127  8.2%    0.70     4.37         48.08
      2           Low Vol / Flat   880 56.8%    0.24     0.48         20.78
      3  Med Vol / Backwardation   148  9.5%    0.36     1.27         49.39
      4 High Vol / Backwardation     5  0.3%    0.91    10.71         80.16
      5  Med Vol / Backwardation   176 11.4%    0.39     1.29         30.55


In [31]:
# ─── Regime-Conditional Propagation Analysis ───

print("\nREGIME-CONDITIONAL RESPONSE TO SPREAD WIDENING")
print("=" * 80)

# For each regime, compute the average forward Brent return conditional on high spread z-score
regime_propagation = []

for regime_id, regime_name in regime_names.items():
    regime_mask = unified['regime'] == regime_id
    regime_data = unified[regime_mask]

    # High-spread days within this regime
    wide_in_regime = regime_data[regime_data['spread_zscore_60d'] > 1.0]

    if len(wide_in_regime) < 5:
        continue

    for horizon in [1, 5, 10, 20]:
        forward_ret = unified['brent_c1'].pct_change(horizon).shift(-horizon)
        rets = forward_ret.loc[forward_ret.index.isin(wide_in_regime.index)].dropna()

        if len(rets) < 3:
            continue

        regime_propagation.append({
            'Regime': regime_name,
            'Horizon': f't+{horizon}',
            'N': len(rets),
            'Avg_Return': f'{rets.mean():.4f}',
            'Hit_Rate': f'{(rets > 0).mean():.1%}',
            'Avg_Vol': regime_data['brent_rvol_20d'].mean(),
        })

regime_prop_df = pd.DataFrame(regime_propagation)
if len(regime_prop_df) > 0:
    print(regime_prop_df.to_string(index=False))
else:
    print("Insufficient data for regime-conditional analysis")


REGIME-CONDITIONAL RESPONSE TO SPREAD WIDENING
                  Regime Horizon   N Avg_Return Hit_Rate  Avg_Vol
 Med Vol / Backwardation     t+1  39     0.0029    53.8% 0.348208
 Med Vol / Backwardation     t+5  39     0.0039    59.0% 0.348208
 Med Vol / Backwardation    t+10  39    -0.0024    51.3% 0.348208
 Med Vol / Backwardation    t+20  39     0.0282    66.7% 0.348208
High Vol / Backwardation     t+1  43     0.0118    62.8% 0.701608
High Vol / Backwardation     t+5  43     0.0145    44.2% 0.701608
High Vol / Backwardation    t+10  43    -0.0122    41.9% 0.701608
High Vol / Backwardation    t+20  43    -0.0083    32.6% 0.701608
          Low Vol / Flat     t+1 275     0.0020    57.8% 0.235860
          Low Vol / Flat     t+5 275     0.0107    62.9% 0.235860
          Low Vol / Flat    t+10 275     0.0235    64.0% 0.235860
          Low Vol / Flat    t+20 275     0.0311    65.1% 0.235860
 Med Vol / Backwardation     t+1  52     0.0061    59.6% 0.363824
 Med Vol / Backwardation    

In [32]:
# ─── Regime Visualization ───

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

regime_colors = plt.cm.Set2(np.linspace(0, 1, len(regime_names)))
color_map = {r: regime_colors[i] for i, r in enumerate(sorted(regime_names.keys()))}

# Panel 1: Regime timeline
ax = axes[0, 0]
for r in sorted(regime_names.keys()):
    mask = unified['regime'] == r
    dates = unified.index[mask]
    for d in dates:
        ax.axvline(x=d, color=color_map[r], alpha=0.3, linewidth=0.3)
ax.plot(unified.index, unified['brent_c1'], color='black', linewidth=0.5)

# Create legend
from matplotlib.patches import Patch
legend_patches = [Patch(color=color_map[r], label=f'R{r}: {regime_names[r]}') for r in sorted(regime_names.keys())]
ax.legend(handles=legend_patches, fontsize=8)
ax.set_title('Brent Price with Regime Overlay')
ax.set_ylabel('$/bbl')

# Panel 2: Regime scatter (vol vs spread)
ax = axes[0, 1]
for r in sorted(regime_names.keys()):
    mask = regime_features['regime'] == r
    ax.scatter(regime_features.loc[mask, 'brent_rvol_20d'],
               regime_features.loc[mask, 'go_brent_spread'],
               c=[color_map[r]], alpha=0.3, s=10, label=f'R{r}: {regime_names[r]}')
ax.set_xlabel('Brent 20d RVol')
ax.set_ylabel('GO-Brent Spread')
ax.set_title('Regime Clusters')
ax.legend(fontsize=8)

# Panel 3: Rolling correlation by regime
ax = axes[1, 0]
roll_corr_60d = unified['spread_chg'].rolling(60).corr(unified['brent_ret'])
for r in sorted(regime_names.keys()):
    mask = unified['regime'] == r
    ax.scatter(unified.index[mask], roll_corr_60d[mask],
               c=[color_map[r]], alpha=0.3, s=5)
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.set_title('60d Rolling Corr (Spread Δ, Brent Ret) by Regime')
ax.set_ylabel('Correlation')

# Panel 4: Brent M1-M2 by regime
ax = axes[1, 1]
for r in sorted(regime_names.keys()):
    mask = unified['regime'] == r
    vals = unified.loc[mask, 'brent_M1_M2'].dropna()
    ax.hist(vals, bins=30, alpha=0.5, color=color_map[r], label=f'R{r}', density=True)
ax.set_xlabel('M1-M2 Spread')
ax.set_ylabel('Density')
ax.set_title('Brent Calendar Spread Distribution by Regime')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('regime_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation — Regime Analysis

**Trader's Perspective:**
- The market operates in distinct regimes with fundamentally different dynamics
- The GO–Brent feedback relationship (if any) may only be active in specific regimes
- High-volatility regimes typically see faster and larger reactions but also more noise
- Backwardation vs contango context is critical — the same spread level means different things in different curve structures
- Regime-conditional hit rates reveal when the signal is reliable vs when it's noise

**Trading Implication:** Any strategy based on GO–Brent signals should be regime-aware

---
# ANALYSIS 8: Failure Analysis

We identify cases where GO–Brent widened significantly but Brent did NOT respond as the 
feedback hypothesis would predict, and study what made these cases different.

In [33]:
# ─── Classify Events as Successes vs Failures ───

# "Success" = Brent rose by at least $1 within 10 trading days after peak spread
# "Failure" = Brent did not rise by $1 within 10 trading days

SUCCESS_THRESHOLD = 1.0  # $/bbl
SUCCESS_HORIZON = 10  # trading days

classifications = []

for _, evt in propagation.iterrows():
    eid = int(evt['event_id'])
    brent_chg_10d = evt.get(f'brent_chg_t{SUCCESS_HORIZON}', np.nan)
    max_brent_chg = max([evt.get(f'brent_chg_t{h}', -np.inf) for h in horizons if h <= SUCCESS_HORIZON])

    is_success = max_brent_chg >= SUCCESS_THRESHOLD

    t0 = evt['t0_date']
    t0_idx = unified.index.get_loc(t0) if t0 in unified.index else None

    # Context features
    context = {}
    if t0_idx is not None and t0_idx > 20:
        pre = unified.iloc[t0_idx - 20:t0_idx]
        context['pre_vol_20d'] = pre['brent_ret'].std() * np.sqrt(252) if 'brent_ret' in pre.columns else np.nan
        context['pre_brent_trend'] = unified.iloc[t0_idx]['brent_c1'] - pre['brent_c1'].mean()
        context['pre_M1M2'] = unified.iloc[t0_idx].get('brent_M1_M2', np.nan)
        context['regime'] = unified.iloc[t0_idx].get('regime_name', 'Unknown')
        context['spread_zscore'] = unified.iloc[t0_idx].get('spread_zscore_60d', np.nan)
        context['gasoil_momentum'] = (
            unified.iloc[t0_idx]['gasoil_c1'] / pre['gasoil_c1'].mean() - 1
        ) if 'gasoil_c1' in pre.columns else np.nan

    classifications.append({
        'Event': eid,
        'Date': t0.date(),
        'Type': 'SUCCESS' if is_success else 'FAILURE',
        'Max_Brent_Chg_10d': max_brent_chg,
        **context,
    })

class_df = pd.DataFrame(classifications)

n_success = (class_df['Type'] == 'SUCCESS').sum()
n_failure = (class_df['Type'] == 'FAILURE').sum()

print(f"EVENT CLASSIFICATION (threshold: Brent +${SUCCESS_THRESHOLD} within {SUCCESS_HORIZON}d)")
print("=" * 80)
print(f"Successes: {n_success} ({n_success/len(class_df):.0%})")
print(f"Failures:  {n_failure} ({n_failure/len(class_df):.0%})")
print()
print(class_df.to_string(index=False))

EVENT CLASSIFICATION (threshold: Brent +$1.0 within 10d)
Successes: 18 (60%)
Failures:  12 (40%)

 Event       Date    Type  Max_Brent_Chg_10d  pre_vol_20d  pre_brent_trend  pre_M1M2                   regime  spread_zscore  gasoil_momentum
     1 2021-05-20 SUCCESS           5.603462     0.256383        -3.134205  0.066993           Low Vol / Flat       2.407482        -0.024258
     2 2021-09-03 SUCCESS           2.967333     0.304569         2.193149  0.618667  Med Vol / Backwardation       2.945965         0.052411
     3 2021-10-15 FAILURE           0.747857     0.179277         4.354736  0.705000           Low Vol / Flat       2.241800         0.076390
     4 2022-02-04 SUCCESS           2.400000     0.212336         5.441060  1.690000  Med Vol / Backwardation       2.930477         0.102279
     5 2022-03-08 FAILURE         -16.841587     0.490727        29.205166  4.477143 High Vol / Backwardation       5.706422         0.712698
     6 2022-03-24 SUCCESS           1.521429     0

In [34]:
# ─── Compare Success vs Failure Characteristics ───

if n_success > 0 and n_failure > 0:
    print("\nSUCCESS vs FAILURE COMPARISON")
    print("=" * 80)

    success = class_df[class_df['Type'] == 'SUCCESS']
    failure = class_df[class_df['Type'] == 'FAILURE']

    comparison_cols = ['pre_vol_20d', 'pre_brent_trend', 'pre_M1M2', 'spread_zscore', 'gasoil_momentum']

    for col in comparison_cols:
        if col in class_df.columns:
            s_vals = success[col].dropna()
            f_vals = failure[col].dropna()

            if len(s_vals) >= 2 and len(f_vals) >= 2:
                t_stat, p_val = stats.ttest_ind(s_vals, f_vals, equal_var=False)
                print(f"  {col:25s}: Success avg={s_vals.mean():.3f}, Failure avg={f_vals.mean():.3f}, "
                      f"t={t_stat:.2f}, p={p_val:.3f} {'*' if p_val < 0.10 else ''}")
            else:
                print(f"  {col:25s}: Insufficient data for comparison")

    # Regime distribution
    print("\n  Regime distribution:")
    if 'regime' in class_df.columns:
        for regime_name in class_df['regime'].unique():
            if pd.isna(regime_name):
                continue
            s_count = (success['regime'] == regime_name).sum()
            f_count = (failure['regime'] == regime_name).sum()
            print(f"    {regime_name:30s}: Success={s_count}, Failure={f_count}")
else:
    print("\nInsufficient data to compare success vs failure characteristics")
    print(f"  (Need both success and failure events; got {n_success} successes, {n_failure} failures)")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

if 'pre_vol_20d' in class_df.columns:
    for idx, col_label in enumerate([('pre_vol_20d', 'Pre-Event Volatility'),
                                      ('pre_brent_trend', 'Pre-Event Brent Trend'),
                                      ('spread_zscore', 'Spread Z-Score at Event')]):
        col, label = col_label
        ax = axes[idx]
        for t, color in [('SUCCESS', COLORS['backwardation']), ('FAILURE', COLORS['contango'])]:
            vals = class_df[class_df['Type'] == t][col].dropna()
            if len(vals) > 0:
                ax.hist(vals, bins=max(5, len(vals)//3), alpha=0.5, color=color, label=t, density=True)
        ax.set_title(label)
        ax.legend()

plt.tight_layout()
plt.savefig('failure_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


SUCCESS vs FAILURE COMPARISON
  pre_vol_20d              : Success avg=0.369, Failure avg=0.323, t=0.56, p=0.580 
  pre_brent_trend          : Success avg=-0.448, Failure avg=4.851, t=-2.01, p=0.060 *
  pre_M1M2                 : Success avg=1.435, Failure avg=1.861, t=-0.46, p=0.655 
  spread_zscore            : Success avg=2.077, Failure avg=2.722, t=-1.85, p=0.080 *
  gasoil_momentum          : Success avg=0.042, Failure avg=0.143, t=-1.76, p=0.103 

  Regime distribution:
    Low Vol / Flat                : Success=9, Failure=8
    Med Vol / Backwardation       : Success=6, Failure=2
    High Vol / Backwardation      : Success=3, Failure=2


### Interpretation — Failure Analysis

**Trader's Perspective:**
- Not all spread widenings lead to Brent rallies — understanding failure conditions is essential for risk management
- Pre-event context (volatility, trend, regime) helps differentiate likely successes from likely failures
- High pre-event volatility may indicate that the spread widening is noise rather than signal
- The regime of the market at the time of the event significantly affects the probability of success

**Practical Application:** Before putting on a GO-Brent spread trade, check the current context against the 
success/failure profiles documented here

---
# ANALYSIS 9: Historical Analogue Engine

For each significant spread widening event, we find the most statistically similar historical 
events and compare their outcomes.

In [35]:
# ─── Build Feature Vectors for Event Similarity ───

def build_event_features(data, event_date, lookback=20):
    """Build a feature vector characterizing market conditions around an event."""
    idx = data.index.get_loc(event_date) if event_date in data.index else None
    if idx is None or idx < lookback:
        return None

    pre = data.iloc[idx - lookback:idx]
    current = data.iloc[idx]

    features = {}

    # Spread characteristics
    features['spread_level'] = current.get('go_brent_spread', np.nan)
    features['spread_zscore'] = current.get('spread_zscore_60d', np.nan)
    features['spread_momentum_5d'] = current.get('spread_momentum_5d', np.nan)

    # Price levels and trends
    features['brent_level'] = current.get('brent_c1', np.nan)
    features['brent_20d_return'] = (current.get('brent_c1', np.nan) / pre['brent_c1'].iloc[0] - 1) if 'brent_c1' in pre.columns else np.nan

    # Volatility
    features['brent_vol'] = current.get('brent_rvol_20d', np.nan)

    # Curve structure
    features['M1_M2'] = current.get('brent_M1_M2', np.nan)

    # Gasoil momentum
    features['gasoil_20d_return'] = (current.get('gasoil_c1', np.nan) / pre['gasoil_c1'].iloc[0] - 1) if 'gasoil_c1' in pre.columns else np.nan

    return features

# Build feature matrix for all events
event_features = {}
for _, evt in event_db.iterrows():
    eid = int(evt['event_id'])
    feats = build_event_features(unified, evt['peak_date'])
    if feats is not None:
        event_features[eid] = feats

feat_df = pd.DataFrame(event_features).T
feat_df.index.name = 'event_id'

print(f"Event feature matrix: {feat_df.shape}")
print(feat_df.to_string())

Event feature matrix: (30, 8)
          spread_level  spread_zscore  spread_momentum_5d  brent_level  brent_20d_return  brent_vol      M1_M2  gasoil_20d_return
event_id                                                                                                                         
1             7.081905       2.407482            0.432209    65.011538         -0.023872   0.268117   0.066993           0.012030
2             9.479299       2.945965            0.790847    72.526667          0.012595   0.303096   0.618667           0.041901
3            14.876544       2.241800            0.591023    84.855000          0.118279   0.175997   0.705000           0.162603
4            21.077204       2.930477            1.613762    92.670000          0.132608   0.215313   1.690000           0.190171
5            79.817488       5.706422           31.347017   129.377143          0.416917   0.481973   4.477143           0.888081
6            47.234449       1.963154            5.410061   

In [36]:
# ─── Compute Similarity Matrix ───

# Standardize features for distance computation
feat_clean = feat_df.dropna(axis=1, how='any').dropna(axis=0, how='any')

if len(feat_clean) >= 3:
    scaler_events = StandardScaler()
    feat_scaled = scaler_events.fit_transform(feat_clean)

    # Euclidean distance matrix
    from scipy.spatial.distance import cdist
    dist_matrix = cdist(feat_scaled, feat_scaled, metric='euclidean')

    # Convert to similarity (1 / (1 + distance))
    sim_matrix = 1 / (1 + dist_matrix)
    np.fill_diagonal(sim_matrix, 0)  # Zero self-similarity

    sim_df = pd.DataFrame(sim_matrix, index=feat_clean.index, columns=feat_clean.index)

    # For each event, find most similar events
    print("ANALOGUE LIBRARY")
    print("=" * 80)

    analogue_library = []

    for eid in feat_clean.index:
        similarities = sim_df.loc[eid].sort_values(ascending=False)

        # Get this event's outcome
        evt_row = propagation[propagation['event_id'] == eid]
        if len(evt_row) == 0:
            continue

        this_outcome = evt_row.iloc[0].get('brent_chg_t10', np.nan)
        this_date = evt_row.iloc[0]['t0_date']

        print(f"\nEvent {eid} ({this_date.date()}) — Brent 10d chg: ${this_outcome:.2f}")
        print(f"  Most similar events:")

        for analogue_id, sim_score in similarities.head(3).items():
            if analogue_id == eid:
                continue
            ana_row = propagation[propagation['event_id'] == analogue_id]
            if len(ana_row) == 0:
                continue
            ana_outcome = ana_row.iloc[0].get('brent_chg_t10', np.nan)
            ana_date = ana_row.iloc[0]['t0_date']

            print(f"    E{analogue_id} ({ana_date.date()}): sim={sim_score:.3f}, Brent 10d: ${ana_outcome:.2f}")

            analogue_library.append({
                'Source_Event': eid,
                'Source_Date': this_date.date(),
                'Analogue_Event': analogue_id,
                'Analogue_Date': ana_date.date(),
                'Similarity': sim_score,
                'Source_Outcome': this_outcome,
                'Analogue_Outcome': ana_outcome,
            })

    analogue_df = pd.DataFrame(analogue_library)

    # Correlation between source and analogue outcomes
    if len(analogue_df) >= 5:
        valid = analogue_df.dropna(subset=['Source_Outcome', 'Analogue_Outcome'])
        corr, p_val = stats.pearsonr(valid['Source_Outcome'], valid['Analogue_Outcome'])
        print(f"\n\nAnalogue Predictive Power:")
        print(f"  Correlation(source outcome, analogue outcome): {corr:.3f} (p={p_val:.4f})")
        print(f"  → {'Analogues have predictive value' if p_val < 0.05 else 'Analogues do NOT have significant predictive value'}")

    # Heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(sim_df, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, 
                xticklabels=[f'E{i}' for i in sim_df.index],
                yticklabels=[f'E{i}' for i in sim_df.index])
    ax.set_title('Event Similarity Matrix')
    plt.tight_layout()
    plt.savefig('event_similarity.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Insufficient events for analogue analysis (need ≥3 events with complete features)")

ANALOGUE LIBRARY

Event 1 (2021-05-20) — Brent 10d chg: $5.60
  Most similar events:
    E21 (2024-11-18): sim=0.541, Brent 10d: $-1.21
    E20 (2024-10-31): sim=0.536, Brent 10d: $-0.78
    E2 (2021-09-03): sim=0.529, Brent 10d: $2.97

Event 2 (2021-09-03) — Brent 10d chg: $2.97
  Most similar events:
    E21 (2024-11-18): sim=0.597, Brent 10d: $-1.21
    E12 (2023-06-18): sim=0.550, Brent 10d: $-1.82
    E1 (2021-05-20): sim=0.529, Brent 10d: $5.60

Event 3 (2021-10-15) — Brent 10d chg: $-0.67
  Most similar events:
    E22 (2025-01-17): sim=0.636, Brent 10d: $-3.95
    E19 (2024-07-02): sim=0.617, Brent 10d: $-1.51
    E4 (2022-02-04): sim=0.481, Brent 10d: $1.04

Event 4 (2022-02-04) — Brent 10d chg: $1.04
  Most similar events:
    E19 (2024-07-02): sim=0.530, Brent 10d: $-1.51
    E22 (2025-01-17): sim=0.485, Brent 10d: $-3.95
    E3 (2021-10-15): sim=0.481, Brent 10d: $-0.67

Event 5 (2022-03-08) — Brent 10d chg: $-20.38
  Most similar events:
    E29 (2026-04-05): sim=0.132, Br

### Interpretation — Historical Analogues

**Trader's Perspective:**
- The analogue engine identifies the most similar historical market conditions to any given spread widening event
- If analogue outcomes correlate with actual outcomes, this can improve real-time forecasting
- The similarity metric captures multi-dimensional market state, not just spread level
- Traders can use this as a "quick lookup" when a new spread widening occurs: find similar past events and study what happened

**Limitation:** Small sample size limits the reliability of analogue-based predictions. Treat as complementary to statistical models.

---
# ANALYSIS 10: Broad Relationship Discovery

We systematically search for statistically significant propagation chains across ALL available 
variables, not just GO → Brent. The data decides which relationships matter.

In [37]:
# ─── Build Comprehensive Variable Set ───

# Select all meaningful derived variables for systematic testing
discovery_vars = []

# Core returns
for col in unified.columns:
    if any(x in col for x in ['_ret', '_chg', 'M1_M2', 'M2_M3', 'M2_M4', 'M1_M6', 
                                'spread_zscore', 'spread_momentum']):
        if unified[col].dropna().shape[0] > 100:
            discovery_vars.append(col)

# Add key level changes
for col in ['go_brent_spread', 'go_brent_ratio']:
    if col in unified.columns:
        unified[f'{col}_chg'] = unified[col] - unified[col].shift(1)
        discovery_vars.append(f'{col}_chg')

print(f"Variables for discovery: {len(discovery_vars)}")
print(discovery_vars)

Variables for discovery: 28
['brent_M1_M2', 'brent_M2_M3', 'brent_M2_M4', 'brent_M1_M6', 'gasoil_M1_M2', 'gasoil_M2_M3', 'gasoil_M1_M6', 'brent_ret', 'gasoil_ret', 'spread_chg', 'spread_ret_pct', 'brent_ret_5d', 'gasoil_ret_5d', 'spread_chg_5d', 'brent_ret_10d', 'gasoil_ret_10d', 'spread_chg_10d', 'brent_ret_20d', 'gasoil_ret_20d', 'spread_chg_20d', 'spread_zscore_60d', 'spread_zscore_120d', 'spread_zscore_252d', 'spread_momentum_5d', 'spread_momentum_10d', 'spread_momentum_20d', 'go_brent_spread_chg', 'go_brent_ratio_chg']


In [38]:
# ─── Systematic Lead-Lag Grid Search ───

# For every pair of variables, compute cross-correlation at lag 1-5
# and Granger causality at lag 1-5

discovery_results = []

# Filter to most important variables to keep computation tractable
key_vars = ['brent_ret', 'gasoil_ret', 'spread_chg', 
            'brent_M1_M2', 'gasoil_M1_M2',
            'spread_momentum_5d', 'go_brent_spread_chg']

# Only use vars that exist
key_vars = [v for v in key_vars if v in unified.columns]

print("SYSTEMATIC RELATIONSHIP DISCOVERY")
print("=" * 80)
print(f"Testing {len(key_vars)} x {len(key_vars)} = {len(key_vars)**2} pairs")
print()

for x_var in key_vars:
    for y_var in key_vars:
        if x_var == y_var:
            continue

        # Align data
        pair_data = unified[[x_var, y_var]].dropna()
        if len(pair_data) < 50:
            continue

        # Cross-correlation at lags 1-5
        best_lag = 1
        best_corr = 0
        for lag in range(1, 6):
            corr = pair_data[x_var].iloc[:-lag].corr(pair_data[y_var].iloc[lag:].reset_index(drop=True))
            if pd.notna(corr) and abs(corr) > abs(best_corr):
                best_corr = corr
                best_lag = lag

        # Granger causality at best lag
        try:
            gc = grangercausalitytests(pair_data[[y_var, x_var]], maxlag=5, verbose=False)
            best_gc_p = min(gc[lag][0]['ssr_ftest'][1] for lag in range(1, 6))
            best_gc_f = max(gc[lag][0]['ssr_ftest'][0] for lag in range(1, 6))
        except:
            best_gc_p = 1.0
            best_gc_f = 0.0

        # Significance
        n = len(pair_data)
        sig_threshold = 2 / np.sqrt(n)

        if abs(best_corr) > sig_threshold or best_gc_p < 0.05:
            discovery_results.append({
                'X (leads)': x_var,
                'Y (follows)': y_var,
                'Best_Lag': best_lag,
                'Cross_Corr': best_corr,
                'Corr_Significant': abs(best_corr) > sig_threshold,
                'GC_F_stat': best_gc_f,
                'GC_P_value': best_gc_p,
                'GC_Significant': best_gc_p < 0.05,
            })

disc_df = pd.DataFrame(discovery_results)

# Sort by combined significance
if len(disc_df) > 0:
    disc_df['combined_score'] = disc_df['Cross_Corr'].abs() + (disc_df['GC_P_value'] < 0.05).astype(float)
    disc_df = disc_df.sort_values('combined_score', ascending=False)

    print("SIGNIFICANT RELATIONSHIPS DISCOVERED")
    print("-" * 80)
    print(disc_df.to_string(index=False, float_format='{:.4f}'.format))

    # Highlight unexpected findings
    expected_pairs = [
        ('spread_chg', 'brent_ret'),
        ('brent_ret', 'spread_chg'),
        ('gasoil_ret', 'brent_ret'),
        ('brent_ret', 'gasoil_ret'),
    ]

    unexpected = disc_df[~disc_df.apply(
        lambda row: (row['X (leads)'], row['Y (follows)']) in expected_pairs, axis=1
    )]

    if len(unexpected) > 0:
        print("\n\n*** UNEXPECTED FINDINGS ***")
        print("-" * 60)
        print(unexpected.to_string(index=False))
else:
    print("No significant relationships discovered at the 5% level")

SYSTEMATIC RELATIONSHIP DISCOVERY
Testing 7 x 7 = 49 pairs



SIGNIFICANT RELATIONSHIPS DISCOVERED
--------------------------------------------------------------------------------
          X (leads)         Y (follows)  Best_Lag  Cross_Corr  Corr_Significant  GC_F_stat  GC_P_value  GC_Significant  combined_score
          brent_ret          gasoil_ret         1           0             False     3.5793      0.0281            True          1.0000
          brent_ret         brent_M1_M2         1           0             False    26.9739      0.0000            True          1.0000
go_brent_spread_chg        gasoil_M1_M2         1           0             False    13.1164      0.0000            True          1.0000
go_brent_spread_chg         brent_M1_M2         1           0             False    21.0967      0.0000            True          1.0000
go_brent_spread_chg           brent_ret         1           0             False     7.7790      0.0005            True          1.0000
 spread_momentum_5d go_brent_spread_chg         1           0           

In [39]:
# ─── Propagation Chain Discovery ───
# Look for multi-step chains: A → B → C

print("\nPROPAGATION CHAIN ANALYSIS")
print("=" * 80)

if len(disc_df) > 0:
    # Build adjacency from significant Granger relationships
    gc_significant = disc_df[disc_df['GC_Significant']]

    print("\nSignificant causal chains (Granger, p<0.05):")

    chains_found = []
    for _, row_a in gc_significant.iterrows():
        a = row_a['X (leads)']
        b = row_a['Y (follows)']

        # Look for B → C
        for _, row_b in gc_significant.iterrows():
            if row_b['X (leads)'] == b and row_b['Y (follows)'] != a:
                c = row_b['Y (follows)']
                total_lag = row_a['Best_Lag'] + row_b['Best_Lag']
                chain = f"{a} →({row_a['Best_Lag']}d)→ {b} →({row_b['Best_Lag']}d)→ {c}"
                chains_found.append({
                    'Chain': chain,
                    'Total_Lag': total_lag,
                    'Leg1_F': row_a['GC_F_stat'],
                    'Leg2_F': row_b['GC_F_stat'],
                })
                print(f"  {chain}  [F1={row_a['GC_F_stat']:.2f}, F2={row_b['GC_F_stat']:.2f}]")

    if len(chains_found) == 0:
        print("  No multi-step propagation chains found")
else:
    print("No significant relationships to build chains from")


PROPAGATION CHAIN ANALYSIS

Significant causal chains (Granger, p<0.05):
  brent_ret →(1d)→ gasoil_ret →(1d)→ spread_momentum_5d  [F1=3.58, F2=40.45]
  brent_ret →(1d)→ gasoil_ret →(1d)→ gasoil_M1_M2  [F1=3.58, F2=10.44]
  brent_ret →(1d)→ gasoil_ret →(1d)→ brent_M1_M2  [F1=3.58, F2=22.94]
  brent_ret →(1d)→ brent_M1_M2 →(1d)→ go_brent_spread_chg  [F1=26.97, F2=9.26]
  brent_ret →(1d)→ brent_M1_M2 →(1d)→ spread_momentum_5d  [F1=26.97, F2=11.44]
  brent_ret →(1d)→ brent_M1_M2 →(1d)→ gasoil_M1_M2  [F1=26.97, F2=18.97]
  brent_ret →(1d)→ brent_M1_M2 →(1d)→ spread_chg  [F1=26.97, F2=9.26]
  go_brent_spread_chg →(1d)→ gasoil_M1_M2 →(1d)→ spread_momentum_5d  [F1=13.12, F2=13.84]
  go_brent_spread_chg →(1d)→ gasoil_M1_M2 →(1d)→ brent_M1_M2  [F1=13.12, F2=59.27]
  go_brent_spread_chg →(1d)→ gasoil_M1_M2 →(1d)→ spread_chg  [F1=13.12, F2=12.98]
  go_brent_spread_chg →(1d)→ gasoil_M1_M2 →(1d)→ gasoil_ret  [F1=13.12, F2=9.59]
  go_brent_spread_chg →(1d)→ brent_M1_M2 →(1d)→ spread_momentum_5d  [F1

### Interpretation — Relationship Discovery

**Trader's Perspective:**
- The systematic search reveals ALL statistically significant lead-lag relationships, not just the hypothesized one
- Calendar spread dynamics (M1-M2 changes) may be as important as the GO-Brent spread itself
- Multi-step propagation chains suggest information flows through specific market variables in sequence
- Unexpected findings deserve the most attention — they may represent genuinely novel trading signals

**Key Insight:** The data reveals which variables actually lead, regardless of economic intuition. 
Trade based on evidence, not narrative.

---
# ANALYSIS 11: Trading Implications

Based ONLY on statistically significant findings from the preceding analyses, we present 
practical, evidence-based observations for commodity traders.

In [40]:
# ─── Compile All Statistically Significant Findings ───

print("EVIDENCE-BASED TRADING IMPLICATIONS")
print("=" * 80)

findings = []

# 1. GO-Brent spread mean reversion
spread = unified['go_brent_spread'].dropna()
adf_result = adfuller(spread)
findings.append({
    'Finding': 'GO-Brent Spread Stationarity',
    'Test': 'ADF',
    'Statistic': f'{adf_result[0]:.3f}',
    'P_value': f'{adf_result[1]:.4f}',
    'Conclusion': 'Mean-reverting' if adf_result[1] < 0.05 else 'Unit root (non-stationary)',
})

print("\n1. SPREAD MEAN REVERSION")
print(f"   ADF test on GO-Brent spread: stat={adf_result[0]:.3f}, p={adf_result[1]:.4f}")
if adf_result[1] < 0.05:
    half_life_est = -np.log(2) / np.log(1 + spread.autocorr(lag=1) - 1) if spread.autocorr(lag=1) < 1 else np.nan
    print(f"   → The spread IS mean-reverting")
    print(f"   → Estimated half-life: ~{abs(int(np.log(2) / -np.log(abs(spread.autocorr(1)))))  if abs(spread.autocorr(1)) < 1 else 'N/A'} days")
    print(f"   → Implication: Extreme spread levels are temporary; fade extreme moves")
else:
    print(f"   → The spread is NOT stationary — mean reversion should not be assumed")

# 2. Conditional return analysis
print("\n2. CONDITIONAL BRENT RETURNS (after spread widening)")

# Quintile analysis
unified['spread_quintile'] = pd.qcut(unified['spread_zscore_60d'].dropna(), q=5, labels=['Q1(low)', 'Q2', 'Q3', 'Q4', 'Q5(wide)'])

forward_5d = unified['brent_c1'].pct_change(5).shift(-5)
forward_10d = unified['brent_c1'].pct_change(10).shift(-10)

quintile_analysis = []
for q in ['Q1(low)', 'Q2', 'Q3', 'Q4', 'Q5(wide)']:
    mask = unified['spread_quintile'] == q
    f5 = forward_5d[mask].dropna()
    f10 = forward_10d[mask].dropna()

    if len(f5) < 10:
        continue

    quintile_analysis.append({
        'Quintile': q,
        'N': len(f5),
        'Avg_5d_Ret': f'{f5.mean():.4f}',
        'Hit_5d': f'{(f5>0).mean():.1%}',
        'Avg_10d_Ret': f'{f10.mean():.4f}',
        'Hit_10d': f'{(f10>0).mean():.1%}',
    })

q_df = pd.DataFrame(quintile_analysis)
print(q_df.to_string(index=False))

EVIDENCE-BASED TRADING IMPLICATIONS

1. SPREAD MEAN REVERSION
   ADF test on GO-Brent spread: stat=-2.058, p=0.2619
   → The spread is NOT stationary — mean reversion should not be assumed

2. CONDITIONAL BRENT RETURNS (after spread widening)
Quintile   N Avg_5d_Ret Hit_5d Avg_10d_Ret Hit_10d
 Q1(low) 302    -0.0065  49.3%     -0.0147   40.7%
      Q2 298     0.0005  53.0%      0.0074   54.9%
      Q3 302     0.0069  52.6%      0.0118   56.0%
      Q4 302     0.0037  55.6%      0.0089   57.0%
Q5(wide) 302     0.0098  59.3%      0.0142   54.6%


In [41]:
# 3. Signal reliability assessment
print("\n3. SIGNAL RELIABILITY ASSESSMENT")
print("-" * 60)

# When has the GO-Brent widening signal been historically reliable?
# Compute rolling hit rate (% of times wide spread → Brent up 10d)

wide_mask = unified['spread_zscore_60d'] > 1.0

if wide_mask.sum() > 10:
    wide_dates = unified.index[wide_mask]
    brent_fwd_10d = unified['brent_c1'].pct_change(10).shift(-10)

    hits = brent_fwd_10d.loc[wide_dates].dropna()
    overall_hit = (hits > 0).mean()

    print(f"   Overall hit rate (wide spread → Brent up 10d): {overall_hit:.1%} ({len(hits)} observations)")

    # Binomial test: is this significantly different from 50%?
    binom_result = stats.binomtest(int((hits > 0).sum()), n=len(hits), p=0.5, alternative='greater')
    print(f"   Binomial test (H0: hit rate = 50%): p = {binom_result.pvalue:.4f}")
    print(f"   → {'Statistically significant above 50%' if binom_result.pvalue < 0.05 else 'NOT significantly above 50%'}")

    # Average return comparison
    unconditional_avg = unified['brent_c1'].pct_change(10).dropna().mean()
    conditional_avg = hits.mean()

    print(f"\n   Unconditional 10d Brent return: {unconditional_avg:.4f}")
    print(f"   Conditional 10d Brent return (after wide spread): {conditional_avg:.4f}")
    print(f"   Edge: {conditional_avg - unconditional_avg:.4f}")
else:
    print("   Insufficient wide-spread observations for reliability assessment")

# 4. Confirmation signals
print("\n4. CONFIRMATION SIGNALS")
print("-" * 60)
print("   Variables that improve the signal when combined with wide spread:")

confirmation_vars = ['brent_M1_M2', 'brent_rvol_20d', 'gasoil_M1_M2', 'spread_momentum_5d']
confirmation_vars = [v for v in confirmation_vars if v in unified.columns]

for conf_var in confirmation_vars:
    # Split wide-spread days by confirmation variable (above/below median)
    wide_data = unified[wide_mask].copy()
    if len(wide_data) < 20:
        continue

    median_val = wide_data[conf_var].median()
    above = wide_data[wide_data[conf_var] >= median_val].index
    below = wide_data[wide_data[conf_var] < median_val].index

    hits_above = brent_fwd_10d.loc[brent_fwd_10d.index.isin(above)].dropna()
    hits_below = brent_fwd_10d.loc[brent_fwd_10d.index.isin(below)].dropna()

    if len(hits_above) >= 5 and len(hits_below) >= 5:
        hr_above = (hits_above > 0).mean()
        hr_below = (hits_below > 0).mean()
        print(f"   {conf_var:25s}: High={hr_above:.1%} (n={len(hits_above)}), Low={hr_below:.1%} (n={len(hits_below)}), diff={hr_above-hr_below:+.1%}")


3. SIGNAL RELIABILITY ASSESSMENT
------------------------------------------------------------
   Overall hit rate (wide spread → Brent up 10d): 56.7% (462 observations)
   Binomial test (H0: hit rate = 50%): p = 0.0022
   → Statistically significant above 50%

   Unconditional 10d Brent return: 0.0064
   Conditional 10d Brent return (after wide spread): 0.0122
   Edge: 0.0058

4. CONFIRMATION SIGNALS
------------------------------------------------------------
   Variables that improve the signal when combined with wide spread:
   brent_M1_M2              : High=49.8% (n=231), Low=63.6% (n=231), diff=-13.9%
   brent_rvol_20d           : High=47.6% (n=231), Low=65.8% (n=231), diff=-18.2%
   gasoil_M1_M2             : High=49.4% (n=231), Low=64.1% (n=231), diff=-14.7%
   spread_momentum_5d       : High=51.9% (n=231), Low=61.5% (n=231), diff=-9.5%


In [42]:
# 5. False signal characteristics
print("\n5. FALSE SIGNAL CHARACTERISTICS")
print("-" * 60)
print("   When the spread widened but Brent fell over the next 10 days:")

if wide_mask.sum() > 10:
    false_signals = hits[hits < 0]
    true_signals = hits[hits > 0]

    print(f"   False signals: {len(false_signals)} ({len(false_signals)/len(hits):.1%})")
    print(f"   True signals: {len(true_signals)} ({len(true_signals)/len(hits):.1%})")

    if len(false_signals) >= 3:
        # Context of false signals
        false_context = unified.loc[false_signals.index]
        true_context = unified.loc[true_signals.index]

        context_vars = ['brent_rvol_20d', 'brent_M1_M2', 'spread_zscore_60d']
        context_vars = [v for v in context_vars if v in unified.columns]

        for var in context_vars:
            f_mean = false_context[var].mean()
            t_mean = true_context[var].mean()
            print(f"   {var:25s}: False avg={f_mean:.3f}, True avg={t_mean:.3f}")

# 6. Historical wait time
print("\n6. HISTORICAL CONFIRMATION TIMING")
print("-" * 60)

if len(propagation) > 0:
    # For events where Brent eventually rose, how many days did traders wait?
    confirmation_delays = []
    for _, evt in propagation.iterrows():
        for h in horizons:
            chg = evt.get(f'brent_chg_t{h}', np.nan)
            if not np.isnan(chg) and chg > 0.5:
                confirmation_delays.append(h)
                break

    if confirmation_delays:
        print(f"   Events where Brent rose > $0.50: {len(confirmation_delays)}/{len(propagation)}")
        print(f"   Average wait for confirmation: {np.mean(confirmation_delays):.1f} days")
        print(f"   Median wait: {np.median(confirmation_delays):.1f} days")
        print(f"   Range: {min(confirmation_delays)}-{max(confirmation_delays)} days")
    else:
        print("   No events with meaningful Brent increases found")

# 7. Monitoring variables
print("\n7. VARIABLES TO MONITOR POST-WIDENING")
print("-" * 60)
print("   Based on Granger causality and cross-correlation significance:")

if len(disc_df) > 0:
    # Variables that lead Brent returns
    brent_leaders = disc_df[
        (disc_df['Y (follows)'] == 'brent_ret') & 
        (disc_df['GC_Significant'] | disc_df['Corr_Significant'])
    ].sort_values('GC_F_stat', ascending=False)

    for _, row in brent_leaders.iterrows():
        print(f"   • {row['X (leads)']} → brent_ret (lag {row['Best_Lag']}d, "
              f"corr={row['Cross_Corr']:.3f}, GC p={row['GC_P_value']:.4f})")
else:
    print("   No significant leading indicators found")


5. FALSE SIGNAL CHARACTERISTICS
------------------------------------------------------------
   When the spread widened but Brent fell over the next 10 days:
   False signals: 200 (43.3%)
   True signals: 262 (56.7%)
   brent_rvol_20d           : False avg=0.369, True avg=0.311
   brent_M1_M2              : False avg=1.637, True avg=0.928
   spread_zscore_60d        : False avg=2.035, True avg=1.862

6. HISTORICAL CONFIRMATION TIMING
------------------------------------------------------------
   Events where Brent rose > $0.50: 24/30
   Average wait for confirmation: 3.8 days
   Median wait: 2.0 days
   Range: 1-20 days

7. VARIABLES TO MONITOR POST-WIDENING
------------------------------------------------------------
   Based on Granger causality and cross-correlation significance:
   • brent_M1_M2 → brent_ret (lag 1d, corr=0.000, GC p=0.0000)
   • gasoil_ret → brent_ret (lag 1d, corr=0.000, GC p=0.0001)
   • spread_momentum_5d → brent_ret (lag 1d, corr=0.000, GC p=0.0025)
   • go_b

In [43]:
# ─── Summary Dashboard ───

fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.3)

# Panel 1: Spread history with events
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(unified.index, unified['go_brent_spread'], color=COLORS['spread'], linewidth=0.7)
for _, evt in event_db.iterrows():
    ax1.axvspan(evt['start'], evt['end'], alpha=0.15, color=COLORS['event'])
ax1.axhline(y=mean_spread, color='gray', linestyle=':', alpha=0.5)
ax1.set_title('GO–Brent Spread with Event Detection', fontsize=13)
ax1.set_ylabel('$/bbl')

# Panel 2: Average propagation path
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(x_axis, avg_brent, 'o-', color=COLORS['brent'], linewidth=2, markersize=4)
ax2.fill_between(x_axis, 
                  [a - s for a, s in zip(avg_brent, std_brent)],
                  [a + s for a, s in zip(avg_brent, std_brent)],
                  alpha=0.2, color=COLORS['brent'])
ax2.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax2.set_title('Avg Brent Path Post-Event')
ax2.set_xlabel('Days')
ax2.set_ylabel('$/bbl change')

# Panel 3: Hit ratio
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(horizons, brent_up_pct, 'o-', color=COLORS['brent'], linewidth=2, markersize=5, label='P(Brent ↑)')
ax3.plot(horizons, spread_narrow_pct, 's-', color=COLORS['spread'], linewidth=2, markersize=5, label='P(Spread ↓)')
ax3.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax3.set_title('Hit Ratios')
ax3.set_xlabel('Days')
ax3.legend(fontsize=9)
ax3.set_ylim(0, 1)

# Panel 4: Quintile returns
ax4 = fig.add_subplot(gs[1, 2])
if len(q_df) > 0:
    quintiles = q_df['Quintile'].values
    ret_10d = [float(x) for x in q_df['Avg_10d_Ret'].values]
    colors_bar = [COLORS['contango'] if r < 0 else COLORS['backwardation'] for r in ret_10d]
    ax4.bar(range(len(quintiles)), ret_10d, color=colors_bar, alpha=0.7)
    ax4.set_xticks(range(len(quintiles)))
    ax4.set_xticklabels(quintiles, rotation=45, fontsize=9)
    ax4.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax4.set_title('10d Fwd Return by Spread Quintile')
    ax4.set_ylabel('Avg Return')

# Panel 5: Spread vs forward Brent return scatter
ax5 = fig.add_subplot(gs[2, 0])
x_scatter = unified['spread_zscore_60d'].dropna()
y_scatter = forward_10d.loc[x_scatter.index].dropna()
common_scatter = x_scatter.index.intersection(y_scatter.index)
ax5.scatter(x_scatter.loc[common_scatter], y_scatter.loc[common_scatter], 
            alpha=0.3, s=8, color=COLORS['brent'])
ax5.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax5.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax5.set_xlabel('Spread Z-Score (60d)')
ax5.set_ylabel('Brent 10d Fwd Return')
ax5.set_title('Spread Z-Score vs Forward Brent')

# Panel 6: Regime breakdown
ax6 = fig.add_subplot(gs[2, 1])
if 'regime_name' in unified.columns:
    regime_counts = unified['regime_name'].value_counts()
    ax6.pie(regime_counts, labels=regime_counts.index, autopct='%1.0f%%', 
            textprops={'fontsize': 8})
    ax6.set_title('Market Regime Distribution')

# Panel 7: Key statistics summary
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis('off')
summary_text = "KEY FINDINGS\n" + "="*25 + "\n\n"
summary_text += f"Events detected: {len(event_db)}\n"
if n_success + n_failure > 0:
    summary_text += f"Success rate: {n_success}/{n_success+n_failure} ({n_success/(n_success+n_failure):.0%})\n"
if len(loop_df) > 0:
    summary_text += f"Avg delay to Brent peak: {loop_df['Days_to_Peak_Brent'].mean():.0f}d\n"
summary_text += f"Spread is {'mean-reverting' if adf_result[1] < 0.05 else 'non-stationary'}\n"
if wide_mask.sum() > 10:
    summary_text += f"Wide spread hit rate: {overall_hit:.0%}\n"

ax7.text(0.1, 0.9, summary_text, transform=ax7.transAxes, fontsize=11,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.suptitle('GO–Brent Feedback Loop Research Dashboard', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('research_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Key Research Findings for Commodity Traders

This section synthesizes all statistically significant findings from the preceding 11 analyses 
into actionable conclusions. Every statement below is supported by the evidence presented above.

In [44]:
# ─── Automated Summary Generator ───
# Compile findings with statistical backing

print("=" * 80)
print("KEY RESEARCH FINDINGS FOR COMMODITY TRADERS")
print("=" * 80)

print("\n" + "─" * 80)
print("FINDING 1: GO-BRENT SPREAD CHARACTERISTICS")
print("─" * 80)
print(f"• Sample period: {unified.index.min().date()} to {unified.index.max().date()} ({len(unified)} trading days)")
print(f"• Mean spread: ${mean_spread:.2f}/bbl (σ = ${std_spread:.2f})")
print(f"• Stationarity: {'YES (mean-reverting, ADF p=' + f'{adf_result[1]:.4f})' if adf_result[1] < 0.05 else 'NO (ADF p=' + f'{adf_result[1]:.4f})'}")

print("\n" + "─" * 80)
print("FINDING 2: EVENT FREQUENCY AND MAGNITUDE")
print("─" * 80)
print(f"• {len(event_db)} significant widening events detected over {len(unified)} trading days")
print(f"• Average event duration: {event_db['duration_days'].mean():.0f} days")
print(f"• Average spread widening: ${event_db['spread_widening'].mean():.2f}/bbl")

print("\n" + "─" * 80)
print("FINDING 3: POST-WIDENING BRENT RESPONSE")
print("─" * 80)
if len(summary_df) > 0:
    for _, row in summary_df.iterrows():
        sig = ''
        h = row['Horizon']
        # Check if we stored significance
        print(f"• {h}: Avg Brent Δ = ${row['Brent_Avg_Chg']:.2f}, "
              f"P(Brent↑) = {row['Brent_Up_Pct']:.0%}, "
              f"P(Spread↓) = {row['Spread_Narrows_Pct']:.0%}")

print("\n" + "─" * 80)
print("FINDING 4: LEAD-LAG STRUCTURE")
print("─" * 80)
if len(gc_df) > 0:
    sig_gc = gc_df[gc_df['Significant_5pct']].groupby('Direction').agg({'P_value': 'min', 'F_stat': 'max'}).reset_index()
    for _, row in sig_gc.iterrows():
        print(f"• {row['Direction']}: F={row['F_stat']:.2f}, p={row['P_value']:.4f} **")
else:
    print("• No significant Granger causality found at 5% level")

print("\n" + "─" * 80)
print("FINDING 5: FEEDBACK LOOP VERDICT")
print("─" * 80)
print("• The proposed feedback loop (Spread widens → Brent rises → Spread narrows → repeat)")
print("  must be evaluated based on the statistical evidence in Analyses 4, 5, and 6 above.")
print("• Each leg of the proposed loop has been tested independently.")
print("• See the detailed p-values and effect sizes in those sections for the final verdict.")

print("\n" + "─" * 80)
print("FINDING 6: REGIME DEPENDENCE")
print("─" * 80)
if len(regime_prop_df) > 0:
    print("• The GO-Brent relationship varies significantly across market regimes:")
    for _, row in regime_prop_df.iterrows():
        print(f"  {row['Regime']:30s} | {row['Horizon']}: Avg={row['Avg_Return']}, Hit={row['Hit_Rate']}")

print("\n" + "─" * 80)
print("FINDING 7: PRACTICAL TRADING GUIDANCE")
print("─" * 80)
print("• When observing a spread widening, check:")
print("  1. Current market regime (vol level + curve structure)")
print("  2. Speed and magnitude of widening (sharp moves more informative)")
print("  3. Confirmation from calendar spread dynamics")
print("  4. Pre-existing Brent price trend")
print("• Historical wait time for confirmation varies — see Analysis 11 for distributions")
print("• Position sizing should account for the high variance in outcomes")

print("\n" + "─" * 80)
print("LIMITATIONS & CAVEATS")
print("─" * 80)
print("• Analysis period is limited to 2021-2026 for combined GO+Brent data")
print("• Small event sample sizes limit statistical power")
print("• Results may be period-specific (COVID recovery, geopolitics)")
print("• Transaction costs, liquidity, and slippage are not modeled")
print("• Past statistical relationships may not persist in future regimes")
print()
print("=" * 80)
print("END OF RESEARCH NOTEBOOK")
print("=" * 80)

KEY RESEARCH FINDINGS FOR COMMODITY TRADERS

────────────────────────────────────────────────────────────────────────────────
FINDING 1: GO-BRENT SPREAD CHARACTERISTICS
────────────────────────────────────────────────────────────────────────────────
• Sample period: 2021-01-04 to 2026-05-20 (1570 trading days)
• Mean spread: $25.56/bbl (σ = $15.35)
• Stationarity: NO (ADF p=0.2619)

────────────────────────────────────────────────────────────────────────────────
FINDING 2: EVENT FREQUENCY AND MAGNITUDE
────────────────────────────────────────────────────────────────────────────────
• 30 significant widening events detected over 1570 trading days
• Average event duration: 15 days
• Average spread widening: $12.61/bbl

────────────────────────────────────────────────────────────────────────────────
FINDING 3: POST-WIDENING BRENT RESPONSE
────────────────────────────────────────────────────────────────────────────────
• t+1: Avg Brent Δ = $-0.47, P(Brent↑) = 63%, P(Spread↓) = 100%
• t+2: 